# SQL-каталог из pandas DataFrame

Этот notebook принимает подготовленные pandas DataFrame, разбирает SQL через
AST SQLGlot и создаёт построчную и агрегированную статистику. SQL никогда не
исполняется, подключения к Greenplum нет.


## 1. Зависимости

Установите зависимости в окружение текущего Jupyter kernel:

```bash
python -m pip install "pandas>=2,<3" "sqlglot>=25.34,<26"
```

После установки перезапустите kernel, если Jupyter попросит это сделать.


In [ ]:
import re

PANDAS_REQUIREMENT = "pandas>=2,<3"
SQLGLOT_REQUIREMENT = "sqlglot>=25.34,<26"


def dependency_version_is_supported(name, version):
    if not isinstance(version, str):
        return False
    match = re.match(r"^(\d+)\.(\d+)", version)
    if match is None:
        return False
    major, minor = (int(part) for part in match.groups())
    if name == "pandas":
        return major == 2
    if name == "sqlglot":
        return major == 25 and minor >= 34
    raise ValueError(f"Unknown dependency: {name}")


try:
    import pandas as pd
    import sqlglot
except ImportError as error:
    raise RuntimeError(
        "Install pandas>=2,<3 and sqlglot>=25.34,<26 in this Jupyter kernel, "
        "then restart the kernel and run the notebook again."
    ) from error

incompatible_dependencies = [
    f"{name}={getattr(module, '__version__', 'unknown')}"
    for name, module in (("pandas", pd), ("sqlglot", sqlglot))
    if not dependency_version_is_supported(
        name,
        getattr(module, "__version__", None),
    )
]
if incompatible_dependencies:
    raise RuntimeError(
        "Incompatible notebook dependencies: "
        + ", ".join(incompatible_dependencies)
        + f". Install {PANDAS_REQUIREMENT} and {SQLGLOT_REQUIREMENT}, "
        "restart the kernel, recreate the input DataFrames, and run again."
    )

from IPython.display import display


In [ ]:
QUERY_DF_NAME = "my_queries_df"
SCHEMA_DF_NAME = "my_schema_df"  # Set None when schema metadata is unavailable.
DEFAULT_SCHEMA = "public"
OUTPUT_DIR = None
BUILD_HTML = False

NOTEBOOK_INPUT_STATE_NAMES = (
    "input_queries_df",
    "input_schema_df",
    "schema_df",
)
NOTEBOOK_RESULT_STATE_NAMES = (
    "result",
    "row_analysis_df",
    "details_df",
    "aggregate_df",
    "catalog_tables_df",
    "catalog_columns_df",
    "errors_df",
)
NOTEBOOK_RESERVED_DATAFRAME_NAMES = frozenset(
    (
        *NOTEBOOK_INPUT_STATE_NAMES,
        *NOTEBOOK_RESULT_STATE_NAMES,
        "NOTEBOOK_INPUT_STATE_NAMES",
        "NOTEBOOK_RESULT_STATE_NAMES",
        "NOTEBOOK_RESERVED_DATAFRAME_NAMES",
    )
)

for config_name, dataframe_name in (
    ("QUERY_DF_NAME", QUERY_DF_NAME),
    ("SCHEMA_DF_NAME", SCHEMA_DF_NAME),
):
    if dataframe_name in NOTEBOOK_RESERVED_DATAFRAME_NAMES:
        raise ValueError(
            f"{config_name}={dataframe_name!r} is reserved by the notebook."
        )

for state_name in (*NOTEBOOK_INPUT_STATE_NAMES, *NOTEBOOK_RESULT_STATE_NAMES):
    globals().pop(state_name, None)


## 2. Входные DataFrame

До ячейки «Получение входов» создайте DataFrame, имя которого указано в
`QUERY_DF_NAME`. Нужны колонки `query_text` и `query_text_template`;
`query_id` и `source_row_count` опциональны.

Метаданные схемы опциональны. Для них нужны `schema_name` (либо
`table_schema`), `table_name`, `column_name`. Например,
`prod_dds.calendar_date.dt` и `prod_emart.calendar_date.dt` задаются двумя
отдельными строками.


## 3. Видимый код анализатора

Ниже находятся обычные Python-классы и функции. Они используют AST SQLGlot
для lineage, CTE, подзапросов, `IN`, `BETWEEN`, `LIKE`/`ILIKE` и regex.


In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Any, Literal
LineageStatus = Literal['resolved', 'multi_source', 'ambiguous', 'unresolved']
PredicateOrigin = Literal['original_literal', 'null_check']

@dataclass(frozen=True, order=True, slots=True)
class ColumnRef:
    catalog: str | None
    schema: str | None
    table: str
    column: str

    @property
    def qualified_name(self) -> str:
        return '.'.join((part for part in (self.catalog, self.schema, self.table, self.column) if part))

@dataclass(frozen=True, slots=True)
class LineageResult:
    status: LineageStatus
    columns: tuple[ColumnRef, ...] = ()
    reason: str | None = None

    def normalized(self) -> 'LineageResult':
        columns = tuple(sorted(set(self.columns)))
        status: LineageStatus = self.status
        if status == 'resolved' and len(columns) > 1:
            status = 'multi_source'
        return LineageResult(status=status, columns=columns, reason=self.reason)

@dataclass(frozen=True, slots=True)
class QueryRecord:
    query_id: str
    query_text: str
    query_text_template: str
    source_row_count: int = 1

@dataclass(frozen=True, slots=True)
class Occurrence:
    query_id: str
    query_hash: str
    template_hash: str
    source_row_count: int
    lineage: LineageResult
    clause_context: str
    operator_or_function: str
    value_role: str
    raw_literal: str
    extracted_value: str
    pattern_template: str
    pattern_family: str
    pattern_format: str
    regex_features: dict[str, Any]
    ast_path: str

    def to_dict(self) -> dict[str, Any]:
        lineage = self.lineage.normalized()
        only = lineage.columns[0] if len(lineage.columns) == 1 else None
        return {'query_id': self.query_id, 'query_hash': self.query_hash, 'template_hash': self.template_hash, 'source_row_count': self.source_row_count, 'schema_name': only.schema if only else None, 'table_name': only.table if only else None, 'column_name': only.column if only else None, 'base_columns': [column.qualified_name for column in lineage.columns], 'lineage_status': lineage.status, 'lineage_reason': lineage.reason, 'clause_context': self.clause_context, 'operator_or_function': self.operator_or_function, 'value_role': self.value_role, 'raw_literal': self.raw_literal, 'extracted_value': self.extracted_value, 'pattern_template': self.pattern_template, 'pattern_family': self.pattern_family, 'pattern_format': self.pattern_format, 'regex_features': dict(sorted(self.regex_features.items())), 'ast_path': self.ast_path}

@dataclass(frozen=True, slots=True)
class LiteralUsage:
    lineage: LineageResult
    clause_context: str
    operator_or_function: str
    value_role: str
    values: tuple[str, ...]
    pattern_families: tuple[str, ...]
    pattern_formats: tuple[str, ...]
    condition_count: int
    literal_count: int

@dataclass(frozen=True, slots=True)
class PredicateUsage:
    lineage: LineageResult
    clause_context: str
    operator_or_function: str
    value_role: str
    raw_literal: str
    extracted_value: str
    pattern_template: str | None
    pattern_family: str
    pattern_format: str
    regex_features: dict[str, Any]
    ast_path: str
    origin: PredicateOrigin

@dataclass(frozen=True, slots=True)
class AnalysisError:
    query_id: str
    stage: str
    error_type: str
    message: str
    sql_fragment: str

    def to_dict(self) -> dict[str, str]:
        return {'query_id': self.query_id, 'stage': self.stage, 'error_type': self.error_type, 'message': self.message, 'sql_fragment': self.sql_fragment}

@dataclass(slots=True)
class AnalysisResult:
    occurrences: list[Occurrence] = field(default_factory=list)
    errors: list[AnalysisError] = field(default_factory=list)


In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Iterable, Mapping, Protocol

def normalize_identifier(value: str | None) -> str | None:
    return value.casefold() if value else None

@dataclass(frozen=True, order=True, slots=True)
class TableRef:
    catalog: str | None
    schema: str | None
    table: str
    columns: frozenset[str] | None = None

    @property
    def qualified_name(self) -> str:
        return '.'.join((part for part in (self.catalog, self.schema, self.table) if part))

class SchemaProvider(Protocol):
    default_schema: str | None

    def resolve_table(self, table: str, *, schema: str | None=None, catalog: str | None=None) -> tuple[TableRef, ...]:
        ...

    def has_column(self, table: TableRef, column: str) -> bool | None:
        ...

class MappingSchemaProvider:
    """Cached schema snapshot backed by ``schema -> table -> columns`` mappings."""

    def __init__(self, mapping: Mapping[str, Mapping[str, Iterable[str]]], *, default_schema: str | None=None, catalog: str | None=None) -> None:
        self.default_schema = normalize_identifier(default_schema)
        self.catalog = normalize_identifier(catalog)
        self._tables: dict[tuple[str | None, str, str], TableRef] = {}
        for schema_name, tables in mapping.items():
            normalized_schema = normalize_identifier(schema_name)
            assert normalized_schema is not None
            for table_name, columns in tables.items():
                normalized_table = normalize_identifier(table_name)
                assert normalized_table is not None
                ref = TableRef(catalog=self.catalog, schema=normalized_schema, table=normalized_table, columns=frozenset((column.casefold() for column in columns)))
                self._tables[self.catalog, normalized_schema, normalized_table] = ref

    def resolve_table(self, table: str, *, schema: str | None=None, catalog: str | None=None) -> tuple[TableRef, ...]:
        normalized_table = table.casefold()
        normalized_schema = normalize_identifier(schema) or self.default_schema
        normalized_catalog = normalize_identifier(catalog) or self.catalog
        matches = [ref for (known_catalog, known_schema, known_table), ref in self._tables.items() if known_table == normalized_table and (normalized_schema is None or known_schema == normalized_schema) and (normalized_catalog is None or known_catalog == normalized_catalog)]
        if matches:
            return tuple(sorted(matches))
        if normalized_schema is not None:
            return (TableRef(catalog=normalized_catalog, schema=normalized_schema, table=normalized_table, columns=None),)
        return ()

    def has_column(self, table: TableRef, column: str) -> bool | None:
        if table.columns is None:
            return None
        return column.casefold() in table.columns

    @property
    def tables(self) -> tuple[TableRef, ...]:
        """Return the complete, deterministic inventory behind this snapshot."""
        return tuple(sorted(self._tables.values()))

    def to_snapshot(self) -> dict[str, object]:
        schemas: dict[str, dict[str, list[str]]] = {}
        for table in self.tables:
            if table.schema is None:
                continue
            schemas.setdefault(table.schema, {})[table.table] = sorted(table.columns or ())
        return {'format_version': '1.0', 'catalog_name': self.catalog, 'default_schema': self.default_schema, 'schemas': schemas}


In [ ]:
from __future__ import annotations
import re
from dataclasses import dataclass, field
from typing import Any

@dataclass(frozen=True, slots=True)
class PatternInfo:
    family: str
    regex_features: dict[str, Any] = field(default_factory=dict)

    @property
    def format_signature(self) -> str:
        enabled = [name for name, value in sorted(self.regex_features.items()) if value]
        return '+'.join(enabled) if enabled else self.family

def _unescaped_wildcards(pattern: str) -> list[tuple[int, str]]:
    wildcards: list[tuple[int, str]] = []
    escaped = False
    for index, character in enumerate(pattern):
        if escaped:
            escaped = False
        elif character == '\\':
            escaped = True
        elif character in {'%', '_'}:
            wildcards.append((index, character))
    return wildcards

def _regex_features(pattern: str, *, case_insensitive: bool) -> dict[str, bool]:
    return {'anchored_start': pattern.startswith('^') or pattern.startswith('\\A'), 'anchored_end': pattern.endswith('$') or pattern.endswith('\\Z'), 'groups': bool(re.search('(?<!\\\\)\\(', pattern)), 'character_classes': bool(re.search('(?<!\\\\)\\[', pattern)), 'quantifiers': bool(re.search('(?<!\\\\)(?:[*+?]|\\{\\d+(?:,\\d*)?\\})', pattern)), 'alternation': bool(re.search('(?<!\\\\)\\|', pattern)), 'inline_flags': bool(re.search('\\(\\?[aiLmsux-]+(?::|\\))', pattern)), 'lookaround': any((marker in pattern for marker in ('(?=', '(?!', '(?<=', '(?<!'))), 'backreference': bool(re.search('\\\\(?:[1-9]|g<[^>]+>)', pattern)), 'case_insensitive': case_insensitive}

def classify_pattern(operator: str, raw_value: str) -> PatternInfo:
    normalized = ' '.join(operator.upper().split())
    if 'LIKE' in normalized and 'REGEXP' not in normalized and ('SIMILAR' not in normalized):
        wildcards = _unescaped_wildcards(raw_value)
        if not wildcards:
            family = 'like_exact'
        elif wildcards == [(len(raw_value) - 1, '%')]:
            family = 'like_prefix'
        elif wildcards == [(0, '%')]:
            family = 'like_suffix'
        elif wildcards == [(0, '%'), (len(raw_value) - 1, '%')]:
            family = 'like_contains'
        else:
            family = 'like_complex'
        return PatternInfo(family=family)
    if normalized in {'~', '~*', '!~', '!~*'} or 'REGEXP' in normalized:
        return PatternInfo(family='regex', regex_features=_regex_features(raw_value, case_insensitive=normalized in {'~*', '!~*'} or normalized.endswith('_I')))
    if 'SIMILAR TO' in normalized:
        return PatternInfo(family='similar_to', regex_features=_regex_features(raw_value, case_insensitive=False))
    return PatternInfo(family='exact_value')


In [ ]:
from __future__ import annotations
import re
from dataclasses import dataclass
DEFAULT_PLACEHOLDER = '&CHARACTER'

@dataclass(frozen=True, slots=True)
class PlaceholderMatch:
    value: str
    start: int
    end: int
    matched: bool = True

def extract_placeholder_values(template_value: str, original_value: str, placeholder: str=DEFAULT_PLACEHOLDER) -> tuple[PlaceholderMatch, ...]:
    """Extract literal fragments represented by placeholders.

    SQL has already been parsed at this point. A regular expression is used only
    to align fixed text inside two string-literal values, never to parse SQL.
    """
    if not placeholder or placeholder not in template_value:
        return ()
    fixed_parts = template_value.split(placeholder)
    alignment = '\\A' + '(.*?)'.join((re.escape(part) for part in fixed_parts)) + '\\Z'
    match = re.match(alignment, original_value, flags=re.DOTALL)
    if match is None:
        return (PlaceholderMatch(original_value, 0, len(original_value), matched=False),)
    return tuple((PlaceholderMatch(match.group(index), match.start(index), match.end(index)) for index in range(1, len(fixed_parts))))


In [ ]:
from __future__ import annotations
from collections.abc import Iterable
from sqlglot import exp
from sqlglot.optimizer.scope import Scope, build_scope

class LineageResolver:
    """Resolve SQLGlot columns to physical sources without guessing ambiguity."""

    def __init__(self, expression: exp.Expression, schema: SchemaProvider) -> None:
        root = build_scope(expression)
        if root is None:
            raise ValueError('statement has no query scope')
        self.root = root
        self.schema = schema
        self._column_scopes: dict[int, Scope] = {}
        for scope in root.traverse():
            for column in scope.columns:
                self._column_scopes[id(column)] = scope

    def resolve(self, column: exp.Column) -> LineageResult:
        scope = self._column_scopes.get(id(column), self.root)
        return self._resolve_column(column, scope, set()).normalized()

    def _resolve_column(self, column: exp.Column, scope: Scope | None, visited: set[tuple[int, str]]) -> LineageResult:
        if scope is None:
            return LineageResult('unresolved', reason='column is outside a query scope')
        column_name = column.name.casefold()
        table_alias = column.table.casefold() if column.table else None
        if table_alias:
            search_scope: Scope | None = scope
            while search_scope is not None:
                selected = search_scope.selected_sources.get(table_alias)
                if selected is not None:
                    return self._resolve_source(selected[1], column_name, visited)
                search_scope = search_scope.parent
            return LineageResult('unresolved', reason=f'source alias {table_alias!r} is not visible')
        search_scope = scope
        while search_scope is not None:
            candidates: list[tuple[object, bool | None]] = []
            for _, source in search_scope.selected_sources.values():
                contains = self._source_contains(source, column_name)
                if contains is not False:
                    candidates.append((source, contains))
            known = [source for source, contains in candidates if contains is True]
            possible = known or [source for source, _ in candidates]
            if len(possible) == 1:
                return self._resolve_source(possible[0], column_name, visited)
            if len(possible) > 1:
                merged = self._merge((self._resolve_source(source, column_name, visited) for source in possible), force_ambiguous=True)
                return LineageResult('ambiguous', merged.columns, reason=f'unqualified column {column_name!r} matches multiple sources').normalized()
            search_scope = search_scope.parent
        return LineageResult('unresolved', reason=f'no source contains {column_name!r}')

    def _source_contains(self, source: object, column_name: str) -> bool | None:
        if isinstance(source, exp.Table):
            candidates = self.schema.resolve_table(source.name, schema=source.db or None, catalog=source.catalog or None)
            answers = [self.schema.has_column(candidate, column_name) for candidate in candidates]
            if any((answer is True for answer in answers)):
                return True
            if answers and all((answer is False for answer in answers)):
                return False
            return None
        if isinstance(source, Scope):
            names = self._scope_output_names(source)
            if column_name in names:
                return True
            if '*' in names:
                return None
            return False
        return False

    def _resolve_source(self, source: object, column_name: str, visited: set[tuple[int, str]]) -> LineageResult:
        if isinstance(source, exp.Table):
            return self._resolve_table(source, column_name)
        if isinstance(source, Scope):
            return self._resolve_scope_output(source, column_name, visited)
        return LineageResult('unresolved', reason='unsupported SQL source')

    def _resolve_table(self, table: exp.Table, column_name: str) -> LineageResult:
        candidates = self.schema.resolve_table(table.name, schema=table.db or None, catalog=table.catalog or None)
        eligible = [candidate for candidate in candidates if self.schema.has_column(candidate, column_name) is not False]
        columns = tuple(sorted((ColumnRef(candidate.catalog, candidate.schema, candidate.table, column_name) for candidate in eligible)))
        if len(columns) == 1:
            return LineageResult('resolved', columns)
        if len(columns) > 1:
            return LineageResult('ambiguous', columns, reason='physical table exists in multiple schemas')
        return LineageResult('unresolved', reason=f'table {table.sql()!r} has no column {column_name!r}')

    def _resolve_scope_output(self, scope: Scope, column_name: str, visited: set[tuple[int, str]]) -> LineageResult:
        visit_key = (id(scope), column_name)
        if visit_key in visited:
            return LineageResult('unresolved', reason='recursive lineage cycle')
        next_visited = visited | {visit_key}
        if isinstance(scope.expression, exp.SetOperation):
            if not scope.union_scopes:
                return LineageResult('unresolved', reason='set operation has no branches')
            output_index = self._output_index(scope.union_scopes[0], column_name)
            if output_index is None:
                return LineageResult('unresolved', reason=f'set operation does not export {column_name!r}')
            return self._merge((self._resolve_projection(branch, output_index, next_visited) for branch in scope.union_scopes))
        if not isinstance(scope.expression, exp.Select):
            return LineageResult('unresolved', reason='source is not a SELECT')
        output_index = self._output_index(scope, column_name)
        if output_index is None:
            stars = [projection for projection in scope.expression.expressions if projection.is_star]
            if stars:
                results = []
                for star in stars:
                    table_alias = star.table if isinstance(star, exp.Column) else None
                    synthetic = exp.column(column_name, table=table_alias)
                    results.append(self._resolve_column(synthetic, scope, next_visited))
                return self._merge(results)
            return LineageResult('unresolved', reason=f'source does not export {column_name!r}')
        return self._resolve_projection(scope, output_index, next_visited)

    @staticmethod
    def _scope_output_names(scope: Scope) -> set[str]:
        if scope.outer_columns:
            return {name.casefold() for name in scope.outer_columns}
        expression = scope.expression
        if isinstance(expression, exp.SetOperation) and scope.union_scopes:
            return LineageResolver._scope_output_names(scope.union_scopes[0])
        if isinstance(expression, exp.Select):
            return {projection.alias_or_name.casefold() for projection in expression.expressions if projection.alias_or_name}
        return set()

    @staticmethod
    def _output_index(scope: Scope, column_name: str) -> int | None:
        if scope.outer_columns:
            for index, name in enumerate(scope.outer_columns):
                if name.casefold() == column_name:
                    return index
            return None
        expression = scope.expression
        if not isinstance(expression, exp.Select):
            return None
        for index, projection in enumerate(expression.expressions):
            if projection.alias_or_name.casefold() == column_name:
                return index
        return None

    def _resolve_projection(self, scope: Scope, output_index: int, visited: set[tuple[int, str]]) -> LineageResult:
        expression = scope.expression
        if not isinstance(expression, exp.Select) or output_index >= len(expression.expressions):
            return LineageResult('unresolved', reason='missing set-operation projection')
        projection = expression.expressions[output_index]
        if isinstance(projection, exp.Star):
            return LineageResult('unresolved', reason='star projection needs catalog expansion')
        columns = list(projection.find_all(exp.Column))
        if not columns:
            return LineageResult('unresolved', reason='projection has no source column')
        return self._merge((self._resolve_column(column, scope, visited) for column in columns))

    @staticmethod
    def _merge(results: Iterable[LineageResult], *, force_ambiguous: bool=False) -> LineageResult:
        materialized = list(results)
        columns = tuple(sorted({column for result in materialized for column in result.columns}))
        if force_ambiguous or any((result.status == 'ambiguous' for result in materialized)):
            status = 'ambiguous'
        elif len(columns) > 1:
            status = 'multi_source'
        elif len(columns) == 1:
            status = 'resolved'
        else:
            status = 'unresolved'
        reasons = sorted({result.reason for result in materialized if result.reason})
        return LineageResult(status, columns, '; '.join(reasons) or None).normalized()


In [ ]:
from __future__ import annotations
import hashlib
from dataclasses import dataclass
from functools import lru_cache
from typing import Iterable
import sqlglot
from sqlglot import ErrorLevel, exp

@dataclass(frozen=True, slots=True)
class Operation:
    name: str
    subject: exp.Expression | None
    value_role: str
    classify_as: str | None = None
_BINARY_OPERATORS: tuple[tuple[type[exp.Expression], str], ...] = ((exp.EQ, '='), (exp.NEQ, '!='), (exp.GT, '>'), (exp.GTE, '>='), (exp.LT, '<'), (exp.LTE, '<='), (exp.Like, 'LIKE'), (exp.ILike, 'ILIKE'), (exp.SimilarTo, 'SIMILAR TO'), (exp.RegexpLike, '~'), (exp.RegexpILike, '~*'))

class SQLAnalyzer:

    def __init__(self, schema: SchemaProvider, *, dialect: str='postgres', placeholder: str=DEFAULT_PLACEHOLDER, template_cache_size: int=2048) -> None:
        self.schema = schema
        self.dialect = dialect
        self.placeholder = placeholder
        self._parse_template = lru_cache(maxsize=template_cache_size)(self._parse_template_uncached)

    def analyze_record(self, record: QueryRecord) -> AnalysisResult:
        result = AnalysisResult()
        try:
            originals = self._parse(record.query_text)
            templates = self._parse_template(record.query_text_template)
        except Exception as error:
            result.errors.append(self._error(record, 'parse', error, record.query_text))
            return result
        if len(originals) != len(templates):
            result.errors.append(AnalysisError(query_id=record.query_id, stage='align', error_type='StatementCountMismatch', message=f'original has {len(originals)} statements, template has {len(templates)}', sql_fragment=self._safe_fragment(record.query_text_template)))
            return result
        query_hash = self._hash(record.query_text)
        template_hash = self._hash(record.query_text_template)
        for statement_index, (original, template) in enumerate(zip(originals, templates)):
            try:
                resolver = LineageResolver(original, self.schema)
            except Exception as error:
                result.errors.append(self._error(record, 'lineage', error, original.sql()))
                continue
            original_paths = dict(self._walk_paths(original, f'statement[{statement_index}]'))
            template_paths = list(self._walk_paths(template, f'statement[{statement_index}]'))
            for ast_path, template_node in template_paths:
                if not (isinstance(template_node, exp.Literal) and template_node.is_string and (self.placeholder in str(template_node.this))):
                    continue
                original_node = original_paths.get(ast_path)
                if not isinstance(original_node, exp.Literal) or not original_node.is_string:
                    result.errors.append(AnalysisError(query_id=record.query_id, stage='align', error_type='AstShapeMismatch', message=f'template literal has no original literal at {ast_path}', sql_fragment=self._safe_fragment(template_node.sql())))
                    continue
                operation = self._find_operation(original_node)
                lineage = self._resolve_subject(operation.subject, resolver)
                context = self._clause_context(original_node)
                classification_operator = operation.classify_as or operation.name
                pattern: PatternInfo = classify_pattern(classification_operator, str(original_node.this))
                matches = extract_placeholder_values(str(template_node.this), str(original_node.this), self.placeholder)
                for placeholder_index, match in enumerate(matches):
                    result.occurrences.append(Occurrence(query_id=record.query_id, query_hash=query_hash, template_hash=template_hash, source_row_count=record.source_row_count, lineage=lineage, clause_context=context, operator_or_function=operation.name, value_role=operation.value_role, raw_literal=str(original_node.this), extracted_value=match.value, pattern_template=str(template_node.this), pattern_family=pattern.family, pattern_format=pattern.format_signature, regex_features=pattern.regex_features, ast_path=f'{ast_path}.placeholder[{placeholder_index}]'))
        return result

    def analyze_records(self, records: Iterable[QueryRecord]) -> AnalysisResult:
        combined = AnalysisResult()
        for record in records:
            result = self.analyze_record(record)
            combined.occurrences.extend(result.occurrences)
            combined.errors.extend(result.errors)
        return combined

    def analyze_literal_usages(self, sql: str) -> tuple[LiteralUsage, ...]:
        """Group query literals by predicate, physical lineage and clause context."""
        condition_groups: dict[tuple[int, int, str, str], dict[str, object]] = {}
        for statement_index, statement in enumerate(self._parse(sql)):
            resolver = LineageResolver(statement, self.schema)
            for literal in statement.find_all(exp.Literal):
                owner = self._literal_operation_owner(literal)
                if owner is None:
                    continue
                operation = self._find_operation(literal)
                value_role = 'range' if isinstance(owner, exp.Between) else operation.value_role
                key = (statement_index, id(owner), operation.name, value_role)
                classification_operator = operation.classify_as or operation.name
                raw_value = str(literal.this)
                value = self._literal_display_value(literal, owner)
                pattern = classify_pattern(classification_operator, raw_value)
                group = condition_groups.get(key)
                if group is None:
                    group = {'lineage': self._resolve_subject(operation.subject, resolver), 'context': self._clause_context(literal), 'operator': operation.name, 'role': value_role, 'values': [], 'families': [], 'formats': [], 'literal_count': 0}
                    condition_groups[key] = group
                values = group['values']
                families = group['families']
                formats = group['formats']
                assert isinstance(values, list)
                assert isinstance(families, list)
                assert isinstance(formats, list)
                if value not in values:
                    values.append(value)
                if pattern.family not in families:
                    families.append(pattern.family)
                if pattern.format_signature not in formats:
                    formats.append(pattern.format_signature)
                group['literal_count'] = int(group['literal_count']) + 1
        aggregated: dict[tuple[object, ...], LiteralUsage] = {}
        for group in condition_groups.values():
            lineage = group['lineage']
            assert isinstance(lineage, LineageResult)
            values = tuple((str(value) for value in group['values']))
            families = tuple((str(value) for value in group['families']))
            formats = tuple((str(value) for value in group['formats']))
            signature = (lineage.status, lineage.columns, lineage.reason, group['context'], group['operator'], group['role'], values, families, formats)
            existing = aggregated.get(signature)
            literal_count = int(group['literal_count'])
            if existing is None:
                aggregated[signature] = LiteralUsage(lineage=lineage, clause_context=str(group['context']), operator_or_function=str(group['operator']), value_role=str(group['role']), values=values, pattern_families=families, pattern_formats=formats, condition_count=1, literal_count=literal_count)
            else:
                aggregated[signature] = LiteralUsage(lineage=existing.lineage, clause_context=existing.clause_context, operator_or_function=existing.operator_or_function, value_role=existing.value_role, values=existing.values, pattern_families=existing.pattern_families, pattern_formats=existing.pattern_formats, condition_count=existing.condition_count + 1, literal_count=existing.literal_count + literal_count)
        context_order = {'WHERE': 0, 'JOIN_ON': 1, 'HAVING': 2, 'CASE': 3, 'SELECT': 4, 'OTHER': 5}
        return tuple(sorted(aggregated.values(), key=lambda usage: (0 if usage.lineage.status == 'resolved' else 1, context_order.get(usage.clause_context, 9), tuple((column.qualified_name for column in usage.lineage.columns)), usage.operator_or_function, usage.values)))

    def analyze_predicate_usages(self, sql: str, *, include_literals: bool=True, include_null_checks: bool=True) -> tuple[PredicateUsage, ...]:
        """Return individual predicate values in source order.

        Unlike ``analyze_literal_usages``, this representation is intended for
        row-level output. Repeated AST literals that form one logical boundary
        expression (for example ``1200 + 11``) are emitted once.
        """
        usages: list[PredicateUsage] = []
        for statement_index, statement in enumerate(self._parse(sql)):
            resolver = LineageResolver(statement, self.schema)
            paths = {id(node): path for path, node in self._walk_paths(statement, f'statement[{statement_index}]')}
            seen: set[tuple[object, ...]] = set()
            if include_literals:
                for literal in statement.find_all(exp.Literal):
                    owner = self._literal_operation_owner(literal)
                    if owner is None:
                        continue
                    operation = self._find_operation(literal)
                    value_role = 'range' if isinstance(owner, exp.Between) else operation.value_role
                    raw_value = str(literal.this)
                    display_value = self._predicate_display_value(literal, owner)
                    dedupe_key = (id(owner), operation.name, value_role, display_value)
                    if dedupe_key in seen:
                        continue
                    seen.add(dedupe_key)
                    classification_operator = operation.classify_as or operation.name
                    pattern = classify_pattern(classification_operator, raw_value)
                    usages.append(PredicateUsage(lineage=self._resolve_subject(operation.subject, resolver), clause_context=self._clause_context(literal), operator_or_function=operation.name, value_role=value_role, raw_literal=display_value if display_value != raw_value else raw_value, extracted_value=display_value, pattern_template=None, pattern_family=pattern.family, pattern_format=pattern.format_signature, regex_features=pattern.regex_features, ast_path=paths.get(id(literal), ''), origin='original_literal'))
            if include_null_checks:
                for null_predicate in statement.find_all(exp.Is):
                    if not isinstance(null_predicate.args.get('expression'), exp.Null):
                        continue
                    operator = self._negated_name(null_predicate, 'IS NULL', 'IS NOT NULL')
                    subject = null_predicate.args.get('this')
                    if not isinstance(subject, exp.Expression):
                        subject = None
                    usages.append(PredicateUsage(lineage=self._resolve_subject(subject, resolver), clause_context=self._clause_context(null_predicate), operator_or_function=operator, value_role='null_check', raw_literal='NULL', extracted_value='NULL', pattern_template=None, pattern_family='null_check', pattern_format='null_check', regex_features={}, ast_path=paths.get(id(null_predicate), ''), origin='null_check'))
        return tuple(sorted(usages, key=lambda usage: usage.ast_path))

    def _parse(self, sql: str) -> tuple[exp.Expression, ...]:
        return tuple(sqlglot.parse(sql, read=self.dialect, error_level=ErrorLevel.RAISE))

    def _parse_template_uncached(self, sql: str) -> tuple[exp.Expression, ...]:
        return self._parse(sql)

    @staticmethod
    def _literal_operation_owner(literal: exp.Literal) -> exp.Expression | None:
        operation_types = tuple((expression_type for expression_type, _ in _BINARY_OPERATORS))
        node: exp.Expression = literal
        while node.parent is not None:
            node = node.parent
            if isinstance(node, operation_types + (exp.In, exp.Between, exp.RegexpReplace, exp.Substring)):
                return node
            if isinstance(node, exp.Anonymous) and node.name.upper().startswith('REGEXP'):
                return node
            if isinstance(node, exp.Select):
                child: exp.Expression = literal
                while child.parent is not node and child.parent is not None:
                    child = child.parent
                return child if child in node.expressions else None
        return None

    def _literal_display_value(self, literal: exp.Literal, owner: exp.Expression) -> str:
        candidates: list[exp.Expression] = []
        if isinstance(owner, exp.Between):
            candidates.extend((expression for expression in (owner.args.get('low'), owner.args.get('high')) if isinstance(expression, exp.Expression)))
        elif isinstance(owner, exp.In):
            candidates.extend(owner.expressions)
        elif any((isinstance(owner, expression_type) for expression_type, _ in _BINARY_OPERATORS)):
            candidates.extend((expression for expression in (owner.args.get('this'), owner.args.get('expression')) if isinstance(expression, exp.Expression)))
        for expression in candidates:
            if not self._contains(expression, literal):
                continue
            if isinstance(expression, exp.Literal):
                return str(expression.this)
            return expression.sql(dialect=self.dialect)
        return str(literal.this)

    def _predicate_display_value(self, literal: exp.Literal, owner: exp.Expression) -> str:
        if any((isinstance(owner, expression_type) for expression_type, _ in _BINARY_OPERATORS)):
            candidates = [expression for expression in (owner.args.get('this'), owner.args.get('expression')) if isinstance(expression, exp.Expression) and self._contains(expression, literal)]
            for candidate in candidates:
                subqueries = list(candidate.find_all(exp.Subquery))
                literal_is_inside_subquery = any((self._contains(subquery, literal) for subquery in subqueries))
                if subqueries and (not literal_is_inside_subquery):
                    return str(literal.this)
        return self._literal_display_value(literal, owner)

    @classmethod
    def _walk_paths(cls, expression: exp.Expression, path: str) -> Iterable[tuple[str, exp.Expression]]:
        yield (path, expression)
        keys = list(expression.args)
        if 'with' in keys:
            keys.insert(0, keys.pop(keys.index('with')))
        for key in keys:
            value = expression.args[key]
            if isinstance(value, exp.Expression):
                yield from cls._walk_paths(value, f'{path}.{key}')
            elif isinstance(value, list):
                for index, child in enumerate(value):
                    if isinstance(child, exp.Expression):
                        yield from cls._walk_paths(child, f'{path}.{key}[{index}]')

    @staticmethod
    def _contains(container: exp.Expression | list[exp.Expression] | None, target: exp.Expression) -> bool:
        if container is None:
            return False
        if isinstance(container, list):
            return any((SQLAnalyzer._contains(child, target) for child in container))
        return container is target or any((node is target for node in container.walk()))

    def _find_operation(self, literal: exp.Literal) -> Operation:
        node: exp.Expression | None = literal.parent
        while node is not None:
            if isinstance(node, exp.In) and self._contains(node.args.get('expressions'), literal):
                return Operation(self._negated_name(node, 'IN', 'NOT IN'), node.this, 'list_value')
            if isinstance(node, exp.Between):
                if self._contains(node.args.get('low'), literal):
                    return Operation('BETWEEN', node.this, 'range_low', classify_as='=')
                if self._contains(node.args.get('high'), literal):
                    return Operation('BETWEEN', node.this, 'range_high', classify_as='=')
            for expression_type, operator_name in _BINARY_OPERATORS:
                if isinstance(node, expression_type):
                    left = node.args.get('this')
                    right = node.args.get('expression')
                    if self._contains(right, literal):
                        subject = left
                    elif self._contains(left, literal):
                        subject = right
                    else:
                        continue
                    negative_name = {'LIKE': 'NOT LIKE', 'ILIKE': 'NOT ILIKE', 'SIMILAR TO': 'NOT SIMILAR TO', '~': '!~', '~*': '!~*'}.get(operator_name, f'NOT {operator_name}')
                    role = 'pattern' if operator_name in {'LIKE', 'ILIKE', 'SIMILAR TO', '~', '~*'} else 'comparison_value'
                    return Operation(self._negated_name(node, operator_name, negative_name), subject, role)
            if isinstance(node, exp.RegexpReplace):
                if self._contains(node.args.get('expression'), literal):
                    modifiers = node.args.get('modifiers')
                    case_insensitive = isinstance(modifiers, exp.Literal) and 'i' in str(modifiers.this).casefold()
                    return Operation('REGEXP_REPLACE', node.this, 'regex_pattern', classify_as='REGEXP_I' if case_insensitive else None)
                if self._contains(node.args.get('replacement'), literal):
                    return Operation('REGEXP_REPLACE', node.this, 'replacement_value', classify_as='=')
            if isinstance(node, exp.Substring):
                if self._contains(node.args.get('start'), literal):
                    if literal.is_string:
                        return Operation('SUBSTRING_REGEX', node.this, 'regex_pattern', classify_as='REGEXP')
                    return Operation('SUBSTRING_START', node.this, 'start_position', classify_as='=')
                if self._contains(node.args.get('length'), literal):
                    return Operation('SUBSTRING_LENGTH', node.this, 'length_value', classify_as='=')
            if isinstance(node, exp.Anonymous) and node.name.upper().startswith('REGEXP'):
                arguments = list(node.expressions)
                subject = arguments[0] if arguments else None
                role = 'regex_pattern' if len(arguments) > 1 and self._contains(arguments[1], literal) else 'function_value'
                flags = arguments[2] if len(arguments) > 2 else None
                case_insensitive = isinstance(flags, exp.Literal) and 'i' in str(flags.this).casefold()
                classify_as = 'REGEXP_I' if role == 'regex_pattern' and case_insensitive else None
                if role != 'regex_pattern':
                    classify_as = '='
                return Operation(node.name.upper(), subject, role, classify_as=classify_as)
            node = node.parent
        projection: exp.Expression = literal
        while projection.parent is not None and (not isinstance(projection.parent, exp.Select)):
            projection = projection.parent
        if isinstance(projection.parent, exp.Select):
            return Operation('SELECT_LITERAL', projection, 'literal', classify_as='=')
        return Operation('LITERAL', None, 'literal', classify_as='=')

    @staticmethod
    def _negated_name(node: exp.Expression, positive: str, negative: str) -> str:
        parent = node.parent
        while isinstance(parent, exp.Paren):
            parent = parent.parent
        return negative if isinstance(parent, exp.Not) else positive

    @staticmethod
    def _clause_context(expression: exp.Expression) -> str:
        node = expression.parent
        while node is not None:
            if isinstance(node, exp.Case):
                return 'CASE'
            if isinstance(node, exp.Where):
                return 'WHERE'
            if isinstance(node, exp.Join):
                return 'JOIN_ON'
            if isinstance(node, exp.Having):
                return 'HAVING'
            if isinstance(node, exp.Select):
                return 'SELECT'
            node = node.parent
        return 'OTHER'

    @staticmethod
    def _resolve_subject(subject: exp.Expression | None, resolver: LineageResolver) -> LineageResult:
        if subject is None:
            return LineageResult('unresolved', reason='literal has no subject expression')
        columns = list(subject.find_all(exp.Column))
        if isinstance(subject, exp.Column) and subject not in columns:
            columns.insert(0, subject)
        if not columns:
            return LineageResult('unresolved', reason='subject expression has no columns')
        results = [resolver.resolve(column) for column in columns]
        base_columns = tuple(sorted({base_column for result in results for base_column in result.columns}))
        if any((result.status == 'ambiguous' for result in results)):
            status = 'ambiguous'
        elif len(base_columns) > 1:
            status = 'multi_source'
        elif len(base_columns) == 1:
            status = 'resolved'
        else:
            status = 'unresolved'
        reasons = sorted({result.reason for result in results if result.reason})
        return LineageResult(status, base_columns, '; '.join(reasons) or None).normalized()

    @staticmethod
    def _hash(value: str) -> str:
        return hashlib.sha256(value.encode('utf-8')).hexdigest()

    @staticmethod
    def _safe_fragment(sql: str, limit: int=240) -> str:
        compact = ' '.join(sql.split())
        redacted: list[str] = []
        index = 0
        while index < len(compact):
            if compact[index] == '$':
                delimiter_end = compact.find('$', index + 1)
                if delimiter_end != -1:
                    tag = compact[index + 1:delimiter_end]
                    valid_tag = not tag or ((tag[0].isalpha() or tag[0] == '_') and all((character.isalnum() or character == '_' for character in tag)))
                    if valid_tag:
                        delimiter = compact[index:delimiter_end + 1]
                        closing = compact.find(delimiter, delimiter_end + 1)
                        redacted.append(f'{delimiter}…{delimiter}')
                        index = closing + len(delimiter) if closing != -1 else len(compact)
                        continue
            if compact[index] != "'":
                redacted.append(compact[index])
                index += 1
                continue
            redacted.append("'…'")
            index += 1
            while index < len(compact):
                if compact[index] != "'":
                    index += 1
                elif index + 1 < len(compact) and compact[index + 1] == "'":
                    index += 2
                else:
                    index += 1
                    break
        return ''.join(redacted)[:limit]

    def _error(self, record: QueryRecord, stage: str, error: Exception, sql: str) -> AnalysisError:
        return AnalysisError(query_id=record.query_id, stage=stage, error_type=type(error).__name__, message=self._safe_fragment(str(error).splitlines()[0]), sql_fragment=self._safe_fragment(sql))


In [ ]:
from __future__ import annotations
from collections import Counter
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import TYPE_CHECKING, Any, Iterable, Mapping
if TYPE_CHECKING:
    pass
else:
    CorpusComplexity = Any
FORMAT_VERSION = '1.0'
DEFAULT_EXAMPLE_LIMIT = 5
DEFAULT_TOP_LIMIT = 20

def _ordered_counts(counter: Counter[str]) -> dict[str, int]:
    return {key: value for key, value in sorted(counter.items(), key=lambda item: (-item[1], item[0]))}

def _generated_at(value: str | None) -> str:
    if value is not None:
        return value
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace('+00:00', 'Z')

@dataclass(frozen=True, slots=True)
class _EventValue:
    value: str
    raw_literal: str
    pattern_template: str | None
    pattern_family: str
    pattern_format: str
    regex_features: Mapping[str, Any]

@dataclass(frozen=True, slots=True)
class _UsageEvent:
    query_id: str
    source_row_count: int
    lineage: LineageResult
    clause_context: str
    operator: str
    value_role: str
    values: tuple[_EventValue, ...]
    condition_count: int = 1
    literal_count: int = 1

@dataclass(slots=True)
class _ValueAccumulator:
    value: str
    pattern_family: str
    pattern_format: str
    pattern_template: str | None
    regex_features: dict[str, Any]
    condition_count: int = 0
    source_row_count: int = 0
    query_ids: set[str] = field(default_factory=set)
    raw_examples: set[str] = field(default_factory=set)
    contexts: Counter[str] = field(default_factory=Counter)
    operators: Counter[str] = field(default_factory=Counter)

    def to_dict(self, *, example_limit: int) -> dict[str, Any]:
        return {'value': self.value, 'pattern_family': self.pattern_family, 'pattern_format': self.pattern_format, 'pattern_template': self.pattern_template, 'regex_features': dict(sorted(self.regex_features.items())), 'condition_count': self.condition_count, 'source_row_count': self.source_row_count, 'distinct_query_count': len(self.query_ids), 'context_counts': _ordered_counts(self.contexts), 'operator_counts': _ordered_counts(self.operators), 'raw_examples': sorted(self.raw_examples)[:example_limit], 'example_query_ids': sorted(self.query_ids)[:example_limit]}

@dataclass(slots=True)
class _ColumnAccumulator:
    ref: ColumnRef
    condition_count: int = 0
    literal_count: int = 0
    source_row_count: int = 0
    weighted_literal_count: int = 0
    query_ids: set[str] = field(default_factory=set)
    contexts: Counter[str] = field(default_factory=Counter)
    operators: Counter[str] = field(default_factory=Counter)
    pattern_families: Counter[str] = field(default_factory=Counter)
    pattern_formats: Counter[str] = field(default_factory=Counter)
    values: dict[tuple[str, str, str, str | None], _ValueAccumulator] = field(default_factory=dict)

    def add(self, event: _UsageEvent) -> None:
        weighted_conditions = event.condition_count * event.source_row_count
        self.condition_count += event.condition_count
        self.literal_count += event.literal_count
        self.source_row_count += weighted_conditions
        self.weighted_literal_count += event.literal_count * event.source_row_count
        self.query_ids.add(event.query_id)
        self.contexts[event.clause_context] += weighted_conditions
        self.operators[event.operator] += weighted_conditions
        for value in event.values:
            self.pattern_families[value.pattern_family] += weighted_conditions
            self.pattern_formats[value.pattern_format] += weighted_conditions
            key = (value.value, value.pattern_family, value.pattern_format, value.pattern_template)
            accumulator = self.values.get(key)
            if accumulator is None:
                accumulator = _ValueAccumulator(value=value.value, pattern_family=value.pattern_family, pattern_format=value.pattern_format, pattern_template=value.pattern_template, regex_features=dict(value.regex_features))
                self.values[key] = accumulator
            accumulator.condition_count += event.condition_count
            accumulator.source_row_count += weighted_conditions
            accumulator.query_ids.add(event.query_id)
            accumulator.raw_examples.add(value.raw_literal)
            accumulator.contexts[event.clause_context] += weighted_conditions
            accumulator.operators[event.operator] += weighted_conditions

    def value_rows(self, *, top_limit: int, example_limit: int) -> list[dict[str, Any]]:
        values = sorted(self.values.values(), key=lambda item: (-item.source_row_count, -item.condition_count, item.value, item.pattern_family, item.pattern_format))
        return [item.to_dict(example_limit=example_limit) for item in values[:top_limit]]

    def to_dict(self, *, top_limit: int, example_limit: int) -> dict[str, Any]:
        top_values = self.value_rows(top_limit=top_limit, example_limit=example_limit)
        return {'catalog_name': self.ref.catalog, 'schema_name': self.ref.schema, 'table_name': self.ref.table, 'column_name': self.ref.column, 'qualified_name': self.ref.qualified_name, 'usage_status': 'active' if self.condition_count else 'unused', 'condition_count': self.condition_count, 'literal_count': self.literal_count, 'source_row_count': self.source_row_count, 'weighted_literal_count': self.weighted_literal_count, 'distinct_query_count': len(self.query_ids), 'context_counts': _ordered_counts(self.contexts), 'operator_counts': _ordered_counts(self.operators), 'pattern_family_counts': _ordered_counts(self.pattern_families), 'pattern_format_counts': _ordered_counts(self.pattern_formats), 'top_values': top_values, 'top_patterns': [row for row in top_values if row['pattern_family'] != 'exact_value'], 'example_query_ids': sorted(self.query_ids)[:example_limit]}

@dataclass(slots=True)
class _QualityAccumulator:
    status: str
    columns: tuple[str, ...]
    reason: str | None
    context: str
    operator: str
    value_role: str
    values: tuple[str, ...]
    condition_count: int = 0
    literal_count: int = 0
    source_row_count: int = 0
    query_ids: set[str] = field(default_factory=set)

    def add(self, event: _UsageEvent) -> None:
        self.condition_count += event.condition_count
        self.literal_count += event.literal_count
        self.source_row_count += event.condition_count * event.source_row_count
        self.query_ids.add(event.query_id)

    def to_dict(self, *, example_limit: int) -> dict[str, Any]:
        return {'lineage_status': self.status, 'base_columns': list(self.columns), 'lineage_reason': self.reason, 'clause_context': self.context, 'operator_or_function': self.operator, 'value_role': self.value_role, 'values': list(self.values), 'condition_count': self.condition_count, 'literal_count': self.literal_count, 'source_row_count': self.source_row_count, 'distinct_query_count': len(self.query_ids), 'example_query_ids': sorted(self.query_ids)[:example_limit]}

@dataclass(frozen=True, slots=True)
class CatalogReport:
    metadata: Mapping[str, Any]
    summary: Mapping[str, Any]
    tables: tuple[Mapping[str, Any], ...]
    quality: Mapping[str, Any]

    def to_dict(self) -> dict[str, Any]:
        return {'metadata': dict(self.metadata), 'summary': dict(self.summary), 'tables': [dict(table) for table in self.tables], 'quality': dict(self.quality)}

    def column_rows(self) -> list[dict[str, Any]]:
        return [dict(column) for table in self.tables for column in table.get('columns', [])]

def _column_from_qualified_name(value: str, *, default_schema: str | None) -> ColumnRef | None:
    parts = [part.casefold() for part in value.split('.') if part]
    if len(parts) >= 4:
        return ColumnRef(parts[-4], parts[-3], parts[-2], parts[-1])
    if len(parts) == 3:
        return ColumnRef(None, parts[0], parts[1], parts[2])
    if len(parts) == 2:
        return ColumnRef(None, default_schema, parts[0], parts[1])
    return None

def _inventory(schema: MappingSchemaProvider) -> dict[ColumnRef, _ColumnAccumulator]:
    accumulators: dict[ColumnRef, _ColumnAccumulator] = {}
    for table in schema.tables:
        for column in sorted(table.columns or ()):
            ref = ColumnRef(table.catalog, table.schema, table.table, column)
            accumulators[ref] = _ColumnAccumulator(ref)
    return accumulators

def _build_report(events: Iterable[_UsageEvent], schema: MappingSchemaProvider, *, source_label: str, source_commit: str | None, dialect: str, query_count: int | None, parsed_query_count: int | None, generated_at: str | None, top_limit: int, example_limit: int) -> CatalogReport:
    columns = _inventory(schema)
    quality_groups: dict[tuple[Any, ...], _QualityAccumulator] = {}
    status_counts: Counter[str] = Counter()
    seen_query_ids: set[str] = set()
    for event in events:
        seen_query_ids.add(event.query_id)
        normalized = event.lineage.normalized()
        weighted_conditions = event.condition_count * event.source_row_count
        status_counts[normalized.status] += weighted_conditions
        if normalized.status == 'resolved' and len(normalized.columns) == 1:
            ref = normalized.columns[0]
            accumulator = columns.get(ref)
            if accumulator is None:
                accumulator = _ColumnAccumulator(ref)
                columns[ref] = accumulator
            accumulator.add(event)
            continue
        column_names = tuple((column.qualified_name for column in normalized.columns))
        values = tuple((value.value for value in event.values))
        key = (normalized.status, column_names, normalized.reason, event.clause_context, event.operator, event.value_role, values)
        group = quality_groups.get(key)
        if group is None:
            group = _QualityAccumulator(status=normalized.status, columns=column_names, reason=normalized.reason, context=event.clause_context, operator=event.operator, value_role=event.value_role, values=values)
            quality_groups[key] = group
        group.add(event)
    table_accumulators: dict[tuple[str | None, str | None, str], list[_ColumnAccumulator]] = {}
    for accumulator in columns.values():
        key = (accumulator.ref.catalog, accumulator.ref.schema, accumulator.ref.table)
        table_accumulators.setdefault(key, []).append(accumulator)
    table_rows: list[dict[str, Any]] = []
    for (catalog, schema_name, table_name), table_columns in sorted(table_accumulators.items(), key=lambda item: tuple((part or '' for part in item[0]))):
        table_columns.sort(key=lambda item: item.ref.column)
        table_queries = {query for column in table_columns for query in column.query_ids}
        contexts: Counter[str] = Counter()
        operators: Counter[str] = Counter()
        families: Counter[str] = Counter()
        for column in table_columns:
            contexts.update(column.contexts)
            operators.update(column.operators)
            families.update(column.pattern_families)
        qualified_name = '.'.join((part for part in (catalog, schema_name, table_name) if part))
        table_rows.append({'catalog_name': catalog, 'schema_name': schema_name, 'table_name': table_name, 'qualified_name': qualified_name, 'column_count': len(table_columns), 'active_column_count': sum((column.condition_count > 0 for column in table_columns)), 'condition_count': sum((column.condition_count for column in table_columns)), 'literal_count': sum((column.literal_count for column in table_columns)), 'source_row_count': sum((column.source_row_count for column in table_columns)), 'distinct_query_count': len(table_queries), 'context_counts': _ordered_counts(contexts), 'operator_counts': _ordered_counts(operators), 'pattern_family_counts': _ordered_counts(families), 'columns': [column.to_dict(top_limit=top_limit, example_limit=example_limit) for column in table_columns]})
    quality_rows = [group.to_dict(example_limit=example_limit) for group in sorted(quality_groups.values(), key=lambda group: (-group.source_row_count, group.status, group.columns, group.context, group.operator, group.values))]
    total_columns = len(columns)
    active_columns = sum((column.condition_count > 0 for column in columns.values()))
    resolved_source_rows = status_counts.get('resolved', 0)
    all_source_rows = sum(status_counts.values())
    quality_source_rows = all_source_rows - resolved_source_rows
    summary = {'query_count': query_count if query_count is not None else len(seen_query_ids), 'parsed_query_count': parsed_query_count if parsed_query_count is not None else len(seen_query_ids), 'table_count': len(table_rows), 'column_count': total_columns, 'active_column_count': active_columns, 'unused_column_count': total_columns - active_columns, 'resolved_condition_count': sum((column.condition_count for column in columns.values())), 'resolved_literal_count': sum((column.literal_count for column in columns.values())), 'resolved_source_row_count': resolved_source_rows, 'quality_source_row_count': quality_source_rows, 'lineage_status_counts': _ordered_counts(status_counts), 'lineage_resolution_rate': resolved_source_rows / all_source_rows if all_source_rows else 1.0}
    quality = {'group_count': len(quality_rows), 'source_row_count': quality_source_rows, 'groups': quality_rows}
    metadata = {'format_version': FORMAT_VERSION, 'source_label': source_label, 'source_commit': source_commit, 'dialect': dialect, 'generated_at': _generated_at(generated_at), 'top_limit': top_limit, 'example_limit': example_limit}
    return CatalogReport(metadata, summary, tuple(table_rows), quality)

def _corpus_events(corpus: CorpusComplexity) -> Iterable[_UsageEvent]:
    for query in corpus.queries:
        for usage in query.literal_usages:
            values: list[_EventValue] = []
            for raw_value in usage.values:
                pattern = classify_pattern(usage.operator_or_function, raw_value)
                family = pattern.family
                pattern_format = pattern.format_signature
                regex_features = pattern.regex_features
                if len(usage.pattern_families) == 1:
                    family = usage.pattern_families[0]
                if len(usage.pattern_formats) == 1:
                    pattern_format = usage.pattern_formats[0]
                if family in {'regex', 'similar_to'} and (not regex_features):
                    classifier = 'SIMILAR TO' if family == 'similar_to' else 'REGEXP_I' if 'case_insensitive' in pattern_format else 'REGEXP'
                    regex_features = classify_pattern(classifier, raw_value).regex_features
                values.append(_EventValue(value=raw_value, raw_literal=raw_value, pattern_template=None, pattern_family=family, pattern_format=pattern_format, regex_features=regex_features))
            yield _UsageEvent(query_id=query.name, source_row_count=1, lineage=usage.lineage, clause_context=usage.clause_context, operator=usage.operator_or_function, value_role=usage.value_role, values=tuple(values), condition_count=usage.condition_count, literal_count=usage.literal_count)

def build_catalog_report(corpus: CorpusComplexity, schema: MappingSchemaProvider, *, generated_at: str | None=None, top_limit: int=DEFAULT_TOP_LIMIT, example_limit: int=DEFAULT_EXAMPLE_LIMIT) -> CatalogReport:
    return _build_report(_corpus_events(corpus), schema, source_label=corpus.source_label, source_commit=corpus.source_commit, dialect=corpus.dialect, query_count=corpus.files_seen, parsed_query_count=corpus.files_parsed, generated_at=generated_at, top_limit=top_limit, example_limit=example_limit)

def _detail_events(rows: Iterable[Mapping[str, Any]], schema: MappingSchemaProvider) -> Iterable[_UsageEvent]:
    for index, row in enumerate(rows):
        column_names = list(row.get('base_columns') or [])
        if not column_names and row.get('table_name') and row.get('column_name'):
            parts = [row.get('schema_name') or schema.default_schema, row.get('table_name'), row.get('column_name')]
            column_names = ['.'.join((str(part) for part in parts if part))]
        columns = tuple((column for value in column_names if (column := _column_from_qualified_name(str(value), default_schema=schema.default_schema)) is not None))
        status = str(row.get('lineage_status') or 'unresolved')
        lineage = LineageResult(status, columns, str(row['lineage_reason']) if row.get('lineage_reason') else None).normalized()
        raw_literal = str(row.get('raw_literal') or '')
        extracted_value = str(row.get('extracted_value') or raw_literal)
        operator = str(row.get('operator_or_function') or 'UNKNOWN')
        pattern = classify_pattern(operator, raw_literal)
        family = str(row.get('pattern_family') or pattern.family)
        pattern_format = str(row.get('pattern_format') or pattern.format_signature)
        regex_features = row.get('regex_features')
        if not isinstance(regex_features, Mapping):
            regex_features = pattern.regex_features
        yield _UsageEvent(query_id=str(row.get('query_id') or f'row-{index + 1}'), source_row_count=max(1, int(row.get('source_row_count') or 1)), lineage=lineage, clause_context=str(row.get('clause_context') or 'OTHER'), operator=operator, value_role=str(row.get('value_role') or 'value'), values=(_EventValue(value=extracted_value, raw_literal=raw_literal, pattern_template=str(row['pattern_template']) if row.get('pattern_template') is not None else None, pattern_family=family, pattern_format=pattern_format, regex_features=dict(regex_features)),))

def build_catalog_report_from_details(rows: Iterable[Mapping[str, Any]], schema: MappingSchemaProvider, *, source_label: str='details.jsonl', source_commit: str | None=None, dialect: str='postgres', generated_at: str | None=None, top_limit: int=DEFAULT_TOP_LIMIT, example_limit: int=DEFAULT_EXAMPLE_LIMIT) -> CatalogReport:
    return _build_report(_detail_events(rows, schema), schema, source_label=source_label, source_commit=source_commit, dialect=dialect, query_count=None, parsed_query_count=None, generated_at=generated_at, top_limit=top_limit, example_limit=example_limit)


In [ ]:
from __future__ import annotations
import html
from collections import Counter
from typing import Any, Mapping

def _escape(value: object) -> str:
    return html.escape(str(value), quote=True)

def _format_int(value: object) -> str:
    try:
        return f'{int(value):,}'.replace(',', '\xa0')
    except (TypeError, ValueError):
        return '0'

def _distribution(values: Mapping[str, int], *, empty: str='нет') -> str:
    if not values:
        return f'<span class="empty">{_escape(empty)}</span>'
    maximum = max(values.values(), default=1)
    parts = []
    for label, value in values.items():
        width = max(4, round(value / maximum * 100)) if maximum else 0
        parts.append(f'<span class="dist-item"><span class="dist-label">{_escape(label)}</span><span class="dist-bar"><i style="width:{width}%"></i></span><strong>{_format_int(value)}</strong></span>')
    return ''.join(parts)

def _tokens(values: Mapping[str, int], *, kind: str='neutral') -> str:
    if not values:
        return '<span class="empty">нет</span>'
    return ''.join((f'<span class="token token-{_escape(kind)}"><b>{_escape(label)}</b><small>{_format_int(value)}</small></span>' for label, value in values.items()))

def _value_cards(values: list[Mapping[str, Any]]) -> str:
    if not values:
        return '<p class="empty zero-note">Значения не обнаружены</p>'
    cards = []
    for index, value in enumerate(values, start=1):
        raw_examples = value.get('raw_examples') or []
        templates = []
        if value.get('pattern_template'):
            templates.append(str(value['pattern_template']))
        raw_line = ''
        if raw_examples or templates:
            fragments = []
            if raw_examples:
                fragments.append('SQL: ' + ', '.join((str(item) for item in raw_examples)))
            if templates:
                fragments.append('шаблон: ' + ', '.join(templates))
            raw_line = f"""<span class="value-raw">{_escape(' · '.join(fragments))}</span>"""
        features = [name for name, enabled in (value.get('regex_features') or {}).items() if enabled]
        feature_line = f"""<span class="feature-line">{_escape(', '.join(features))}</span>""" if features else ''
        cards.append(f"""<div class="value-card"><span class="value-rank">{index:02d}</span><span class="value-main"><code>{_escape(value.get('value', ''))}</code><span class="value-family">{_escape(value.get('pattern_family', ''))}</span>{raw_line}{feature_line}</span><span class="value-count"><strong>{_format_int(value.get('source_row_count', 0))}</strong><small>взвеш. использований</small><span>{_format_int(value.get('distinct_query_count', 0))} запросов</span></span></div>""")
    return ''.join(cards)

def _column_row(column: Mapping[str, Any]) -> str:
    active = column.get('usage_status') == 'active'
    contexts = column.get('context_counts') or {}
    patterns = column.get('pattern_family_counts') or {}
    operators = column.get('operator_counts') or {}
    values = column.get('top_values') or []
    search_bits = [column.get('qualified_name', ''), *contexts.keys(), *patterns.keys(), *operators.keys()]
    for value in values:
        search_bits.extend([value.get('value', ''), value.get('pattern_template', '') or '', *(value.get('raw_examples') or [])])
    search_data = ' '.join((str(bit) for bit in search_bits)).casefold()
    context_data = ' '.join(contexts)
    pattern_data = ' '.join(patterns)
    status_label = 'есть связанные значения' if active else 'связанные значения не найдены'
    detail_open = ' open' if active and int(column.get('source_row_count', 0)) >= 20 else ''
    return f'''\n      <div class="column-row {('is-active' if active else 'is-unused')}"\n           data-active="{('active' if active else 'unused')}"\n           data-contexts="{_escape(context_data)}"\n           data-patterns="{_escape(pattern_data)}"\n           data-search="{_escape(search_data)}">\n        <details{detail_open}>\n          <summary class="column-summary">\n            <span class="column-status" title="{_escape(status_label)}"></span>\n            <span class="column-name">\n              <code>{_escape(column.get('qualified_name', ''))}</code>\n              <small>{_escape(status_label)}</small>\n            </span>\n            <span class="column-kpi"><strong>{_format_int(column.get('source_row_count', 0))}</strong><small>использований</small></span>\n            <span class="column-kpi"><strong>{_format_int(column.get('distinct_query_count', 0))}</strong><small>запросов</small></span>\n            <span class="column-kpi"><strong>{_format_int(len(values))}</strong><small>значений в топе</small></span>\n            <span class="column-chevron" aria-hidden="true">⌄</span>\n          </summary>\n          <div class="column-detail">\n            <div class="column-distributions">\n              <section><h4>Секции SQL</h4><div class="distribution">{_distribution(contexts)}</div></section>\n              <section><h4>Операторы</h4><div class="token-list">{_tokens(operators, kind='operator')}</div></section>\n              <section><h4>Типы значений и масок</h4><div class="token-list">{_tokens(patterns, kind='pattern')}</div></section>\n            </div>\n            <section class="values-section">\n              <div class="minor-heading"><h4>Популярные значения, маски и regex</h4><span>до {len(values)} позиций</span></div>\n              <div class="value-list">{_value_cards(values)}</div>\n            </section>\n            <p class="examples"><b>Примеры запросов:</b> {_escape(', '.join(column.get('example_query_ids') or []) or 'нет')}</p>\n          </div>\n        </details>\n      </div>'''

def _table_card(table: Mapping[str, Any], *, index: int) -> str:
    columns = table.get('columns') or []
    open_attribute = ' open' if index < 3 else ''
    return f'''\n    <article class="table-card" data-table="{_escape(table.get('qualified_name', ''))}">\n      <details class="table-details"{open_attribute}>\n        <summary class="table-summary">\n          <span class="table-index">{index + 1:02d}</span>\n          <span class="table-name"><small>физическая таблица</small><code>{_escape(table.get('qualified_name', ''))}</code></span>\n          <span class="table-kpi"><strong>{_format_int(table.get('active_column_count', 0))}</strong><small>активных / {_format_int(table.get('column_count', 0))}</small></span>\n          <span class="table-kpi"><strong>{_format_int(table.get('source_row_count', 0))}</strong><small>использований</small></span>\n          <span class="table-kpi"><strong>{_format_int(table.get('distinct_query_count', 0))}</strong><small>запросов</small></span>\n          <span class="table-chevron" aria-hidden="true">⌄</span>\n        </summary>\n        <div class="table-body">\n          <div class="table-profile">\n            <section><h4>Где используется</h4><div class="distribution">{_distribution(table.get('context_counts') or {})}</div></section>\n            <section><h4>Чем сравнивается</h4><div class="token-list">{_tokens(table.get('operator_counts') or {}, kind='operator')}</div></section>\n            <section><h4>Форматы</h4><div class="token-list">{_tokens(table.get('pattern_family_counts') or {}, kind='pattern')}</div></section>\n          </div>\n          <div class="column-header"><span>Колонка</span><span>Использования</span><span>Запросы</span><span>Топ</span></div>\n          <div class="column-list">{''.join((_column_row(column) for column in columns))}</div>\n        </div>\n      </details>\n    </article>'''

def _quality_rows(quality: Mapping[str, Any]) -> str:
    groups = quality.get('groups') or []
    if not groups:
        return '<p class="quality-clean">Все условия однозначно связаны с физическими колонками.</p>'
    rows = []
    for group in groups:
        columns = group.get('base_columns') or []
        rows.append(f"""<div class="quality-row"><span class="quality-status">{_escape(group.get('lineage_status', ''))}</span><span class="quality-main"><code>{_escape(', '.join(columns) or 'без кандидатов')}</code><span>{_escape(group.get('lineage_reason') or 'причина не указана')}</span><small>{_escape(group.get('clause_context', ''))} · {_escape(group.get('operator_or_function', ''))} · {_escape(', '.join(group.get('values') or []))}</small></span><strong>{_format_int(group.get('source_row_count', 0))}</strong></div>""")
    return ''.join(rows)

def render_catalog_html(report: CatalogReport) -> str:
    payload = report.to_dict()
    metadata = payload['metadata']
    summary = payload['summary']
    quality = payload['quality']
    tables = sorted(payload['tables'], key=lambda table: (-int(table.get('source_row_count', 0)), str(table.get('qualified_name', ''))))
    contexts: Counter[str] = Counter()
    patterns: Counter[str] = Counter()
    for table in tables:
        contexts.update(table.get('context_counts') or {})
        patterns.update(table.get('pattern_family_counts') or {})
    resolution_rate = float(summary.get('lineage_resolution_rate', 0)) * 100
    context_options = ''.join((f'<option value="{_escape(name)}">{_escape(name)} · {_format_int(count)}</option>' for name, count in contexts.most_common()))
    pattern_options = ''.join((f'<option value="{_escape(name)}">{_escape(name)} · {_format_int(count)}</option>' for name, count in patterns.most_common()))
    cards = ''.join((_table_card(table, index=index) for index, table in enumerate(tables)))
    status_tags = ''.join((f'<span><b>{_escape(status)}</b>{_format_int(count)}</span>' for status, count in (summary.get('lineage_status_counts') or {}).items()))
    return f"""<!doctype html>\n<html lang="ru">\n<head>\n  <meta charset="utf-8">\n  <meta name="viewport" content="width=device-width, initial-scale=1">\n  <title>Статистика использования SQL · дата-каталог</title>\n  <style>\n    :root {{\n      --paper:#f1ecdf; --paper-light:#fbf8f0; --paper-deep:#e4dbca;\n      --ink:#1c201d; --muted:#6c6a61; --line:#c7bcaa;\n      --brick:#ad392a; --brick-dark:#78251c; --olive:#5f6c50;\n      --gold:#bd812c; --blue:#3c6571; --shadow:0 16px 38px rgba(53,43,28,.08);\n      --serif:"Iowan Old Style","Palatino Linotype",Georgia,serif;\n      --sans:"Avenir Next",Avenir,"Trebuchet MS",sans-serif;\n      --mono:"SFMono-Regular",Menlo,Monaco,Consolas,monospace;\n    }}\n    * {{ box-sizing:border-box; }}\n    html {{ scroll-behavior:smooth; }}\n    body {{ margin:0; color:var(--ink); background:linear-gradient(rgba(45,39,29,.035) 1px,transparent 1px),var(--paper); background-size:100% 29px; font:15px/1.5 var(--sans); }}\n    code {{ font-family:var(--mono); overflow-wrap:anywhere; }}\n    button,input,select {{ font:inherit; }}\n    button {{ cursor:pointer; }}\n    .masthead {{ position:relative; overflow:hidden; color:#f6efe2; background:var(--ink); border-bottom:7px solid var(--brick); }}\n    .masthead::after {{ content:"CATALOG"; position:absolute; right:-2vw; bottom:-50px; color:rgba(255,255,255,.04); font:900 clamp(90px,18vw,250px)/1 var(--sans); letter-spacing:-.08em; }}\n    .masthead-inner,.page,.toolbar-inner,.footer-inner {{ width:min(1540px,calc(100% - 42px)); margin:0 auto; }}\n    .masthead-inner {{ position:relative; z-index:1; padding:58px 0 46px; }}\n    .eyebrow {{ display:flex; align-items:center; gap:12px; margin:0 0 17px; color:#d9cbb7; font:800 11px/1 var(--sans); letter-spacing:.18em; text-transform:uppercase; }}\n    .eyebrow::before {{ content:""; width:48px; border-top:3px solid var(--brick); }}\n    h1 {{ max-width:1050px; margin:0; font:600 clamp(40px,6.5vw,88px)/.95 var(--serif); letter-spacing:-.045em; }}\n    .dek {{ max-width:900px; margin:23px 0 0; color:#d7cec0; font-size:18px; }}\n    .source-line {{ display:flex; flex-wrap:wrap; gap:10px 28px; margin-top:29px; color:#aaa99f; font:12px/1.5 var(--mono); }}\n    .page {{ padding:38px 0 78px; }}\n    .summary-grid {{ display:grid; grid-template-columns:1.2fr repeat(4,1fr); background:rgba(251,248,240,.9); border:1px solid var(--line); box-shadow:var(--shadow); }}\n    .summary-lead {{ min-height:154px; padding:25px; border-right:1px solid var(--line); }}\n    .summary-lead small,.summary-card small {{ display:block; color:var(--muted); font-size:10px; letter-spacing:.12em; text-transform:uppercase; }}\n    .summary-lead strong {{ display:block; margin:8px 0 12px; color:var(--brick); font:600 42px/1 var(--serif); }}\n    .resolution-track {{ height:8px; overflow:hidden; background:var(--paper-deep); }}\n    .resolution-track i {{ display:block; height:100%; background:var(--olive); }}\n    .status-tags {{ display:flex; flex-wrap:wrap; gap:6px 14px; margin-top:12px; }}\n    .status-tags span {{ color:var(--muted); font-size:10px; }} .status-tags b {{ margin-right:5px; color:var(--ink); }}\n    .summary-card {{ min-height:154px; padding:25px 20px; border-right:1px solid var(--line); }}\n    .summary-card:last-child {{ border-right:0; }}\n    .summary-card strong {{ display:block; margin-top:17px; font:600 43px/1 var(--serif); letter-spacing:-.04em; }}\n    .summary-card span {{ display:block; margin-top:10px; color:var(--muted); font-size:12px; }}\n    .method-note {{ display:grid; grid-template-columns:1fr auto; gap:24px; align-items:center; margin-top:20px; padding:22px 26px; background:rgba(251,248,240,.75); border:1px solid var(--line); border-left:6px solid var(--brick); }}\n    .method-note h2 {{ margin:0 0 6px; font:600 24px/1.1 var(--serif); }} .method-note p {{ margin:0; color:var(--muted); }}\n    .method-note code {{ padding:9px 12px; color:#f7eee2; background:var(--ink); font-size:11px; }}\n    .toolbar {{ position:sticky; top:0; z-index:20; margin-top:30px; color:white; background:rgba(28,32,29,.97); border-bottom:3px solid var(--brick); backdrop-filter:blur(12px); }}\n    .toolbar-inner {{ display:grid; grid-template-columns:minmax(280px,1fr) 190px 190px 210px auto; gap:9px; align-items:center; padding:11px 0; }}\n    .toolbar input,.toolbar select {{ width:100%; min-height:43px; padding:8px 11px; color:#f8f1e6; background:#292d29; border:1px solid #555a52; border-radius:0; outline:none; }}\n    .toolbar input:focus,.toolbar select:focus {{ border-color:#e4a08c; box-shadow:0 0 0 2px rgba(228,160,140,.18); }}\n    #visible-column-count {{ min-width:104px; color:#c9c4ba; text-align:right; font:11px/1.3 var(--mono); }}\n    .catalog-head {{ display:flex; justify-content:space-between; gap:20px; align-items:end; margin:34px 0 16px; }}\n    .catalog-head h2,.quality-section h2 {{ margin:0; font:600 38px/1 var(--serif); letter-spacing:-.03em; }}\n    .catalog-head p {{ max-width:560px; margin:0; color:var(--muted); text-align:right; }}\n    .table-card {{ margin:13px 0; background:rgba(251,248,240,.9); border:1px solid var(--line); border-left:6px solid var(--olive); box-shadow:0 6px 20px rgba(54,42,24,.05); }}\n    .table-card[hidden] {{ display:none; }}\n    summary {{ list-style:none; }} summary::-webkit-details-marker {{ display:none; }}\n    .table-summary {{ display:grid; grid-template-columns:48px minmax(260px,1fr) repeat(3,125px) 28px; gap:14px; align-items:center; min-height:98px; padding:15px 19px; cursor:pointer; }}\n    .table-summary:hover,.column-summary:hover {{ background:rgba(228,219,202,.38); }}\n    .table-index {{ color:var(--brick); font:700 17px/1 var(--mono); }}\n    .table-name small {{ display:block; margin-bottom:7px; color:var(--muted); font-size:9px; letter-spacing:.14em; text-transform:uppercase; }}\n    .table-name code {{ font-size:18px; font-weight:700; }}\n    .table-kpi {{ text-align:right; border-left:1px solid var(--line); padding-left:14px; }} .table-kpi strong {{ display:block; font:600 28px/1 var(--serif); }} .table-kpi small {{ color:var(--muted); font-size:9px; text-transform:uppercase; }}\n    .table-chevron,.column-chevron {{ font-size:24px; transition:transform .2s; }} details[open]>.table-summary .table-chevron,details[open]>.column-summary .column-chevron {{ transform:rotate(180deg); }}\n    .table-body {{ border-top:1px solid var(--line); }}\n    .table-profile {{ display:grid; grid-template-columns:1fr 1fr 1fr; gap:24px; padding:20px 24px; background:rgba(228,219,202,.22); border-bottom:1px solid var(--line); }}\n    h4 {{ margin:0 0 11px; font-size:10px; letter-spacing:.13em; text-transform:uppercase; }}\n    .distribution {{ display:flex; flex-direction:column; gap:6px; }}\n    .dist-item {{ display:grid; grid-template-columns:92px 1fr 35px; gap:8px; align-items:center; font-size:10px; }} .dist-label {{ overflow:hidden; text-overflow:ellipsis; }}\n    .dist-bar {{ height:5px; background:var(--paper-deep); }} .dist-bar i {{ display:block; height:100%; background:var(--brick); }}\n    .token-list {{ display:flex; flex-wrap:wrap; gap:6px; }} .token {{ display:inline-flex; gap:8px; align-items:center; padding:5px 8px; border:1px solid var(--line); background:var(--paper-light); font:10px/1 var(--mono); }} .token small {{ color:var(--muted); }} .token-pattern {{ border-color:#a9b69c; }} .token-operator {{ border-color:#bca688; }}\n    .column-header {{ display:grid; grid-template-columns:minmax(300px,1fr) 140px 140px 100px; gap:12px; padding:10px 73px 10px 51px; color:var(--muted); background:var(--ink); font-size:9px; letter-spacing:.12em; text-transform:uppercase; }}\n    .column-header span:not(:first-child) {{ text-align:right; }}\n    .column-row {{ border-bottom:1px solid var(--line); }} .column-row:last-child {{ border-bottom:0; }} .column-row[hidden] {{ display:none; }}\n    .column-summary {{ display:grid; grid-template-columns:18px minmax(260px,1fr) 140px 140px 100px 24px; gap:12px; align-items:center; min-height:72px; padding:10px 18px; cursor:pointer; }}\n    .column-status {{ width:8px; height:8px; background:var(--olive); border-radius:50%; box-shadow:0 0 0 4px rgba(95,108,80,.12); }} .is-unused .column-status {{ background:#aaa295; box-shadow:none; }}\n    .column-name code {{ display:block; font-size:13px; }} .column-name small {{ display:block; margin-top:3px; color:var(--muted); font-size:9px; }} .is-unused .column-name code {{ color:#77746c; }}\n    .column-kpi {{ text-align:right; }} .column-kpi strong {{ display:block; font:600 22px/1 var(--serif); }} .column-kpi small {{ color:var(--muted); font-size:8px; text-transform:uppercase; }}\n    .column-detail {{ padding:21px 24px 25px 48px; background:#f8f4ea; border-top:1px dashed var(--line); }}\n    .column-distributions {{ display:grid; grid-template-columns:1fr 1fr 1fr; gap:26px; }}\n    .values-section {{ margin-top:22px; }} .minor-heading {{ display:flex; justify-content:space-between; gap:12px; align-items:center; margin-bottom:8px; }} .minor-heading h4 {{ margin:0; }} .minor-heading span {{ color:var(--muted); font-size:9px; }}\n    .value-list {{ border-top:1px solid var(--line); }}\n    .value-card {{ display:grid; grid-template-columns:42px minmax(220px,1fr) 150px; gap:12px; align-items:center; padding:11px 6px; border-bottom:1px solid var(--line); }}\n    .value-rank {{ color:var(--brick); font:700 11px/1 var(--mono); }} .value-main code {{ display:block; font-size:12px; }} .value-family {{ display:inline-block; margin-top:5px; padding:2px 6px; color:var(--brick-dark); background:#ead8cb; font:9px/1.3 var(--mono); }}\n    .value-raw,.feature-line {{ display:block; margin-top:5px; color:var(--muted); font:9px/1.35 var(--mono); }} .feature-line {{ color:var(--blue); }}\n    .value-count {{ text-align:right; }} .value-count strong {{ display:block; font:600 21px/1 var(--serif); }} .value-count small,.value-count span {{ display:block; color:var(--muted); font-size:8px; }}\n    .examples {{ margin:16px 0 0; color:var(--muted); font:10px/1.5 var(--mono); }} .empty {{ color:var(--muted); font-style:italic; font-size:10px; }} .zero-note {{ margin:10px 0; }}\n    .quality-section {{ margin-top:44px; padding:28px; background:rgba(251,248,240,.9); border:1px solid var(--line); border-top:6px solid var(--brick); box-shadow:var(--shadow); }}\n    .quality-intro {{ max-width:850px; color:var(--muted); }} .quality-list {{ margin-top:18px; border-top:1px solid var(--line); }}\n    .quality-row {{ display:grid; grid-template-columns:110px minmax(260px,1fr) 70px; gap:16px; align-items:center; padding:13px 5px; border-bottom:1px solid var(--line); }}\n    .quality-status {{ padding:5px 7px; color:#fff; background:var(--brick); font:9px/1 var(--mono); text-align:center; text-transform:uppercase; }} .quality-main code,.quality-main span,.quality-main small {{ display:block; }} .quality-main code {{ font-size:11px; }} .quality-main span {{ color:var(--muted); font-size:10px; }} .quality-main small {{ margin-top:3px; color:var(--blue); }} .quality-row>strong {{ text-align:right; font:600 24px/1 var(--serif); }}\n    .quality-clean {{ padding:18px; color:var(--olive); background:#edf0e8; }}\n    footer {{ color:#c9c1b5; background:var(--ink); }} .footer-inner {{ display:flex; justify-content:space-between; gap:20px; padding:24px 0; font:10px/1.5 var(--mono); }}\n    @media (max-width:1080px) {{\n      .summary-grid {{ grid-template-columns:repeat(2,1fr); }} .summary-lead {{ grid-column:span 2; border-right:0; border-bottom:1px solid var(--line); }} .summary-card:nth-child(odd) {{ border-right:0; }}\n      .toolbar-inner {{ grid-template-columns:1fr 1fr 1fr; }} .toolbar input {{ grid-column:span 2; }}\n      .table-summary {{ grid-template-columns:40px minmax(220px,1fr) 105px 105px 26px; }} .table-summary .table-kpi:nth-of-type(5) {{ display:none; }}\n      .column-header {{ display:none; }} .column-summary {{ grid-template-columns:18px minmax(220px,1fr) 105px 105px 24px; }} .column-summary .column-kpi:nth-of-type(5) {{ display:none; }}\n    }}\n    @media (max-width:760px) {{\n      .masthead-inner,.page,.toolbar-inner,.footer-inner {{ width:min(100% - 24px,1540px); }} .masthead-inner {{ padding:38px 0 32px; }} .dek {{ font-size:15px; }}\n      .summary-grid {{ display:block; }} .summary-lead,.summary-card {{ min-height:0; border:0; border-bottom:1px solid var(--line); }} .summary-card strong {{ margin-top:8px; }}\n      .method-note {{ display:block; }} .method-note code {{ display:block; margin-top:14px; overflow:auto; }}\n      .toolbar {{ position:relative; }} .toolbar-inner {{ display:grid; grid-template-columns:1fr; }} .toolbar input {{ grid-column:auto; }} #visible-column-count {{ text-align:left; }}\n      .catalog-head {{ display:block; }} .catalog-head p {{ margin-top:10px; text-align:left; }}\n      .table-summary {{ grid-template-columns:34px 1fr 24px; min-height:82px; }} .table-summary .table-kpi {{ display:none; }}\n      .table-profile,.column-distributions {{ grid-template-columns:1fr; }}\n      .column-summary {{ grid-template-columns:14px 1fr 24px; }} .column-summary .column-kpi {{ display:none; }}\n      .column-detail {{ padding:18px 15px; }} .value-card {{ grid-template-columns:30px 1fr; }} .value-count {{ grid-column:2; text-align:left; }}\n      .quality-row {{ grid-template-columns:1fr; }} .quality-row>strong {{ text-align:left; }} .footer-inner {{ display:block; }}\n    }}\n    @media print {{ .toolbar {{ display:none; }} .table-card,.quality-section {{ box-shadow:none; break-inside:avoid; }} body {{ background:#fff; }} details {{ display:block; }} }}\n  </style>\n</head>\n<body>\n  <header class="masthead">\n    <div class="masthead-inner">\n      <p class="eyebrow">SQL lineage · значения · маски · regex</p>\n      <h1>Статистика использования SQL</h1>\n      <p class="dek">Каталожный срез по каждой физической таблице и колонке: где она участвует, чем сравнивается и какие значения или шаблоны встречаются чаще всего.</p>\n      <div class="source-line"><span>источник: {_escape(metadata.get('source_label'))}</span><span>диалект: {_escape(metadata.get('dialect'))}</span><span>формат: {_escape(metadata.get('format_version'))}</span><span>сформировано: {_escape(metadata.get('generated_at'))}</span></div>\n    </div>\n  </header>\n  <main class="page">\n    <section class="summary-grid" aria-label="Сводка">\n      <div class="summary-lead"><small>Однозначность lineage</small><strong>{resolution_rate:.1f}%</strong><div class="resolution-track"><i style="width:{resolution_rate:.2f}%"></i></div><div class="status-tags">{status_tags}</div></div>\n      <div class="summary-card"><small>Таблицы</small><strong>{_format_int(summary.get('table_count'))}</strong><span>полный DDL-инвентарь</span></div>\n      <div class="summary-card"><small>Колонки</small><strong>{_format_int(summary.get('column_count'))}</strong><span>{_format_int(summary.get('active_column_count'))} со значениями</span></div>\n      <div class="summary-card"><small>Запросы</small><strong>{_format_int(summary.get('query_count'))}</strong><span>{_format_int(summary.get('parsed_query_count'))} разобрано</span></div>\n      <div class="summary-card"><small>Условия</small><strong>{_format_int(summary.get('resolved_condition_count'))}</strong><span>с физической колонкой</span></div>\n    </section>\n    <section class="method-note"><div><h2>Как читать результат</h2><p>В популярность колонки попадают только однозначно разрешённые условия. Серые колонки входят в DDL, но не встретились. Неоднозначные случаи сохранены ниже отдельно.</p></div><code>query_text → details.jsonl → catalog JSON → HTML</code></section>\n    <div class="toolbar">\n      <div class="toolbar-inner">\n        <input id="catalog-search" type="search" placeholder="Таблица, колонка, значение, маска или regex…" autocomplete="off">\n        <select id="activity-filter"><option value="all">Все колонки</option><option value="active">С найденными значениями</option><option value="unused">Без найденных значений</option></select>\n        <select id="context-filter"><option value="all">Все секции SQL</option>{context_options}</select>\n        <select id="pattern-filter"><option value="all">Все типы значений</option>{pattern_options}</select>\n        <span id="visible-column-count">{_format_int(summary.get('column_count'))} колонок</span>\n      </div>\n    </div>\n    <div class="catalog-head"><h2>Таблицы и колонки</h2><p>Таблицы отсортированы по частоте использования; внутри сохранён полный алфавитный состав колонок из DDL.</p></div>\n    <section id="catalog-list">{cards}</section>\n    <section class="quality-section" id="quality">\n      <h2>Неоднозначные и неразрешённые случаи</h2>\n      <p class="quality-intro">Эти {_format_int(quality.get('source_row_count'))} взвешенных использований требуют проверки схемы или SQL. Они намеренно не добавлены к популярности ни одной физической колонки.</p>\n      <div class="quality-list">{_quality_rows(quality)}</div>\n    </section>\n  </main>\n  <footer><div class="footer-inner"><span>Данные отчёта агрегированы заранее; SQL в браузере не разбирается.</span><span>{_escape(metadata.get('source_commit') or 'без commit источника')}</span></div></footer>\n  <script>\n    (() => {{\n      const search = document.getElementById('catalog-search');\n      const activity = document.getElementById('activity-filter');\n      const context = document.getElementById('context-filter');\n      const pattern = document.getElementById('pattern-filter');\n      const counter = document.getElementById('visible-column-count');\n      const tables = Array.from(document.querySelectorAll('.table-card'));\n      const normalize = value => value.trim().toLocaleLowerCase('ru');\n      function applyFilters() {{\n        const term = normalize(search.value);\n        let visibleColumns = 0;\n        tables.forEach(table => {{\n          let tableVisible = 0;\n          table.querySelectorAll('.column-row').forEach(column => {{\n            const textMatch = !term || normalize(column.dataset.search).includes(term);\n            const activityMatch = activity.value === 'all' || column.dataset.active === activity.value;\n            const contextMatch = context.value === 'all' || column.dataset.contexts.split(' ').includes(context.value);\n            const patternMatch = pattern.value === 'all' || column.dataset.patterns.split(' ').includes(pattern.value);\n            const visible = textMatch && activityMatch && contextMatch && patternMatch;\n            column.hidden = !visible;\n            if (visible) {{ tableVisible += 1; visibleColumns += 1; }}\n          }});\n          table.hidden = tableVisible === 0;\n          if (term && tableVisible) table.querySelector('.table-details').open = true;\n        }});\n        counter.textContent = `${{visibleColumns.toLocaleString('ru-RU')}} колонок`;\n      }}\n      [search, activity, context, pattern].forEach(control => control.addEventListener('input', applyFilters));\n      applyFilters();\n    }})();\n  </script>\n</body>\n</html>"""


In [ ]:
from __future__ import annotations
import json
from collections.abc import Iterator, Mapping
from pathlib import Path
from typing import Any, TextIO

class JsonlWriter:

    def __init__(self, path: str | Path) -> None:
        self.path = Path(path)
        self._handle: TextIO | None = None

    def __enter__(self) -> 'JsonlWriter':
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self._handle = self.path.open('w', encoding='utf-8')
        return self

    def write(self, payload: Mapping[str, Any]) -> None:
        if self._handle is None:
            raise RuntimeError('JsonlWriter must be used as a context manager')
        serialized = json.dumps(dict(payload), ensure_ascii=False, sort_keys=True, separators=(', ', ': '))
        self._handle.write(serialized + '\n')

    def __exit__(self, exc_type: object, exc: object, traceback: object) -> None:
        if self._handle is not None:
            self._handle.close()
            self._handle = None

def iter_jsonl_records(path: str | Path, *, batch_size: int) -> Iterator[list[QueryRecord]]:
    if batch_size <= 0:
        raise ValueError('batch_size must be positive')
    batch: list[QueryRecord] = []
    with Path(path).open(encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            payload = json.loads(line)
            try:
                record = QueryRecord(query_id=str(payload.get('query_id') or f'line-{line_number}'), query_text=str(payload['query_text']), query_text_template=str(payload['query_text_template']), source_row_count=int(payload.get('source_row_count', 1)))
            except (KeyError, TypeError, ValueError) as error:
                raise ValueError(f'invalid JSONL record at line {line_number}: {error}') from error
            batch.append(record)
            if len(batch) >= batch_size:
                yield batch
                batch = []
    if batch:
        yield batch


In [ ]:
from __future__ import annotations
import hashlib
import importlib
import json
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Iterable, Mapping
DETAIL_COLUMNS = ['query_id', 'query_hash', 'template_hash', 'source_row_count', 'catalog_name', 'schema_name', 'table_name', 'column_name', 'base_columns', 'lineage_status', 'lineage_reason', 'clause_context', 'operator_or_function', 'value_role', 'raw_literal', 'extracted_value', 'pattern_template', 'pattern_family', 'pattern_format', 'regex_features', 'ast_path', 'origin']
ERROR_COLUMNS = ['query_id', 'stage', 'error_type', 'message', 'sql_fragment']
AGGREGATE_GROUP_COLUMNS = ['catalog_name', 'schema_name', 'table_name', 'column_name', 'extracted_value', 'clause_context', 'operator_or_function', 'value_role', 'pattern_family', 'pattern_format']
AGGREGATE_COLUMNS = [*AGGREGATE_GROUP_COLUMNS, 'qualified_name', 'source_row_count', 'occurrence_count', 'distinct_query_count', 'distinct_template_count', 'share_of_column', 'example_query_ids']

@dataclass(slots=True)
class DataFrameAnalysis:
    row_analysis_df: Any
    aggregate_df: Any
    details_df: Any
    errors_df: Any
    catalog_columns_df: Any
    catalog_tables_df: Any
    catalog_report: dict[str, Any]
    artifact_paths: dict[str, Path] = field(default_factory=dict)

def _pandas():
    try:
        return importlib.import_module('pandas')
    except ImportError as error:
        raise RuntimeError('DataFrame analysis requires pandas; install gp-sql-analyzer[notebook]') from error

def schema_from_dataframe(schema_df: Any | None, *, default_schema: str | None=None) -> MappingSchemaProvider:
    if schema_df is None:
        return MappingSchemaProvider({}, default_schema=default_schema)
    required = {'table_schema', 'table_name', 'column_name'}
    missing = sorted(required - set(schema_df.columns))
    if missing:
        raise ValueError('schema_df is missing required columns: ' + ', '.join(missing))
    catalogs: set[str] = set()
    mapping: dict[str, dict[str, list[str]]] = {}
    for _, row in schema_df.iterrows():
        schema_name = str(row['table_schema'])
        table_name = str(row['table_name'])
        column_name = str(row['column_name'])
        mapping.setdefault(schema_name, {}).setdefault(table_name, []).append(column_name)
        if 'table_catalog' in schema_df.columns:
            value = row['table_catalog']
            if value is not None and str(value).casefold() not in {'nan', '<na>'}:
                catalogs.add(str(value))
    if len(catalogs) > 1:
        raise ValueError('schema_df must contain at most one table_catalog')
    catalog = next(iter(catalogs), None)
    return MappingSchemaProvider(mapping, default_schema=default_schema, catalog=catalog)

def _hash(value: str) -> str:
    return hashlib.sha256(value.encode('utf-8')).hexdigest()

def _lineage_fields(lineage) -> dict[str, Any]:
    normalized = lineage.normalized()
    only = normalized.columns[0] if len(normalized.columns) == 1 else None
    return {'catalog_name': only.catalog if only else None, 'schema_name': only.schema if only else None, 'table_name': only.table if only else None, 'column_name': only.column if only else None, 'base_columns': [column.qualified_name for column in normalized.columns], 'lineage_status': normalized.status, 'lineage_reason': normalized.reason}

def _occurrence_row(occurrence: Occurrence) -> dict[str, Any]:
    row = occurrence.to_dict()
    row['catalog_name'] = occurrence.lineage.columns[0].catalog if len(occurrence.lineage.columns) == 1 else None
    row['origin'] = 'template'
    return row

def _predicate_row(usage: PredicateUsage, *, record: QueryRecord) -> dict[str, Any]:
    return {'query_id': record.query_id, 'query_hash': _hash(record.query_text), 'template_hash': _hash(record.query_text_template), 'source_row_count': record.source_row_count, **_lineage_fields(usage.lineage), 'clause_context': usage.clause_context, 'operator_or_function': usage.operator_or_function, 'value_role': usage.value_role, 'raw_literal': usage.raw_literal, 'extracted_value': usage.extracted_value, 'pattern_template': usage.pattern_template, 'pattern_family': usage.pattern_family, 'pattern_format': usage.pattern_format, 'regex_features': dict(sorted(usage.regex_features.items())), 'ast_path': usage.ast_path, 'origin': usage.origin}

def _dedupe_signature(row: Mapping[str, Any]) -> tuple[Any, ...]:
    return (tuple(row.get('base_columns') or []), row.get('lineage_status'), row.get('clause_context'), row.get('operator_or_function'), row.get('raw_literal'))

@dataclass(slots=True)
class _AggregateGroup:
    source_row_count: int = 0
    occurrence_count: int = 0
    query_ids: set[str] = field(default_factory=set)
    template_hashes: set[str] = field(default_factory=set)

def _aggregate_rows(rows: Iterable[Mapping[str, Any]], *, example_limit: int) -> list[dict[str, Any]]:
    groups: dict[tuple[Any, ...], _AggregateGroup] = {}
    column_totals: Counter[tuple[Any, ...]] = Counter()
    for row in rows:
        base_columns = list(row.get('base_columns') or [])
        if row.get('lineage_status') != 'resolved' or len(base_columns) != 1:
            continue
        key = tuple((row.get(column) for column in AGGREGATE_GROUP_COLUMNS))
        weight = int(row.get('source_row_count') or 1)
        group = groups.setdefault(key, _AggregateGroup())
        group.source_row_count += weight
        group.occurrence_count += 1
        group.query_ids.add(str(row.get('query_id') or ''))
        group.template_hashes.add(str(row.get('template_hash') or ''))
        column_key = key[:4]
        column_totals[column_key] += weight
    output: list[dict[str, Any]] = []
    for key, group in groups.items():
        values = dict(zip(AGGREGATE_GROUP_COLUMNS, key))
        qualified_name = '.'.join((str(values[name]) for name in ('catalog_name', 'schema_name', 'table_name', 'column_name') if values[name]))
        total = column_totals[key[:4]]
        output.append({**values, 'qualified_name': qualified_name, 'source_row_count': group.source_row_count, 'occurrence_count': group.occurrence_count, 'distinct_query_count': len(group.query_ids), 'distinct_template_count': len(group.template_hashes), 'share_of_column': group.source_row_count / total if total else 0.0, 'example_query_ids': sorted(group.query_ids)[:example_limit]})
    return sorted(output, key=lambda row: (row['qualified_name'], -row['source_row_count'], str(row['extracted_value']), str(row['clause_context']), str(row['operator_or_function'])))

def _validate_queries(queries_df: Any) -> None:
    required = {'query_text', 'query_text_template'}
    missing = sorted(required - set(queries_df.columns))
    if missing:
        raise ValueError('queries_df is missing required columns: ' + ', '.join(missing))
    if 'source_row_count' not in queries_df.columns:
        return
    for value in queries_df['source_row_count'].tolist():
        try:
            parsed = int(value)
        except (TypeError, ValueError, OverflowError) as error:
            raise ValueError('source_row_count must contain positive integers') from error
        if parsed <= 0 or float(value) != parsed:
            raise ValueError('source_row_count must contain positive integers')

def _write_dataframe_jsonl(dataframe: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    dataframe.to_json(path, orient='records', lines=True, force_ascii=False, date_format='iso')

def _write_artifacts(*, output_dir: Path, row_analysis_df: Any, details_df: Any, errors_df: Any, aggregate_df: Any, report: CatalogReport, schema: MappingSchemaProvider, build_html: bool) -> dict[str, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = {'row_analysis': output_dir / 'row_analysis.jsonl', 'details': output_dir / 'details.jsonl', 'errors': output_dir / 'errors.jsonl', 'aggregate': output_dir / 'aggregate.jsonl', 'catalog_json': output_dir / 'catalog-stats.json', 'catalog_columns': output_dir / 'catalog-columns.jsonl', 'schema': output_dir / 'schema.json'}
    _write_dataframe_jsonl(row_analysis_df, paths['row_analysis'])
    _write_dataframe_jsonl(details_df, paths['details'])
    _write_dataframe_jsonl(errors_df, paths['errors'])
    _write_dataframe_jsonl(aggregate_df, paths['aggregate'])
    paths['catalog_json'].write_text(json.dumps(report.to_dict(), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    with JsonlWriter(paths['catalog_columns']) as writer:
        for row in report.column_rows():
            writer.write(row)
    paths['schema'].write_text(json.dumps(schema.to_snapshot(), ensure_ascii=False, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    if build_html:
        html_path = output_dir / 'catalog-stats.html'
        html_path.write_text(render_catalog_html(report), encoding='utf-8')
        paths['html'] = html_path
    return paths

def analyze_dataframe(queries_df: Any, *, schema_df: Any | None=None, default_schema: str | None=None, dialect: str='postgres', placeholder: str='&CHARACTER', include_original_literals: bool=True, include_null_checks: bool=True, output_dir: str | Path | None=None, build_html: bool=False, top_limit: int=20, example_limit: int=5, source_label: str='pandas.DataFrame') -> DataFrameAnalysis:
    pd = _pandas()
    _validate_queries(queries_df)
    if build_html and output_dir is None:
        raise ValueError('output_dir is required when build_html=True')
    schema = schema_from_dataframe(schema_df, default_schema=default_schema)
    analyzer = SQLAnalyzer(schema, dialect=dialect, placeholder=placeholder)
    row_records: list[dict[str, Any]] = []
    detail_rows: list[dict[str, Any]] = []
    error_rows: list[dict[str, Any]] = []
    for position, (index, source_row) in enumerate(queries_df.iterrows(), start=1):
        source = source_row.to_dict()
        query_id_value = source.get('query_id')
        if query_id_value is None or str(query_id_value).casefold() in {'nan', '<na>'}:
            query_id = str(index) if index is not None else f'row-{position}'
        else:
            query_id = str(query_id_value)
        weight = int(source.get('source_row_count', 1))
        record = QueryRecord(query_id=query_id, query_text=str(source['query_text']), query_text_template=str(source['query_text_template']), source_row_count=weight)
        result = analyzer.analyze_record(record)
        row_errors = [error.to_dict() for error in result.errors]
        template_rows = [_occurrence_row(item) for item in result.occurrences]
        row_details = list(template_rows)
        parse_failed = any((error['stage'] == 'parse' for error in row_errors))
        if not parse_failed and (include_original_literals or include_null_checks):
            try:
                predicate_usages = analyzer.analyze_predicate_usages(record.query_text, include_literals=include_original_literals, include_null_checks=include_null_checks)
            except Exception as error:
                row_errors.append({'query_id': query_id, 'stage': 'predicate', 'error_type': type(error).__name__, 'message': str(error).splitlines()[0][:240], 'sql_fragment': ' '.join(record.query_text.split())[:240]})
            else:
                template_signatures = {_dedupe_signature(row) for row in template_rows}
                for usage in predicate_usages:
                    row = _predicate_row(usage, record=record)
                    if usage.origin == 'original_literal' and _dedupe_signature(row) in template_signatures:
                        continue
                    row_details.append(row)
        status_counts = Counter((str(row['lineage_status']) for row in row_details))
        if parse_failed and (not row_details):
            analysis_status = 'error'
        elif row_errors or any((status != 'resolved' for status in status_counts)):
            analysis_status = 'partial'
        else:
            analysis_status = 'ok'
        row_records.append({**source, 'analysis': row_details, 'analysis_count': len(row_details), 'resolved_count': status_counts['resolved'], 'multi_source_count': status_counts['multi_source'], 'ambiguous_count': status_counts['ambiguous'], 'unresolved_count': status_counts['unresolved'], 'analysis_status': analysis_status, 'analysis_errors': row_errors})
        detail_rows.extend(row_details)
        error_rows.extend(row_errors)
    row_analysis_df = pd.DataFrame(row_records, index=queries_df.index.copy())
    details_df = pd.DataFrame(detail_rows, columns=DETAIL_COLUMNS)
    errors_df = pd.DataFrame(error_rows, columns=ERROR_COLUMNS)
    aggregate_rows = _aggregate_rows(detail_rows, example_limit=example_limit)
    aggregate_df = pd.DataFrame(aggregate_rows, columns=AGGREGATE_COLUMNS)
    report = build_catalog_report_from_details(detail_rows, schema, source_label=source_label, dialect=dialect, top_limit=top_limit, example_limit=example_limit)
    catalog_payload = report.to_dict()
    catalog_columns_df = pd.DataFrame(report.column_rows())
    catalog_tables_df = pd.DataFrame([{key: value for key, value in table.items() if key != 'columns'} for table in catalog_payload['tables']])
    artifact_paths: dict[str, Path] = {}
    if output_dir is not None:
        artifact_paths = _write_artifacts(output_dir=Path(output_dir), row_analysis_df=row_analysis_df, details_df=details_df, errors_df=errors_df, aggregate_df=aggregate_df, report=report, schema=schema, build_html=build_html)
    return DataFrameAnalysis(row_analysis_df=row_analysis_df, aggregate_df=aggregate_df, details_df=details_df, errors_df=errors_df, catalog_columns_df=catalog_columns_df, catalog_tables_df=catalog_tables_df, catalog_report=catalog_payload, artifact_paths=artifact_paths)


In [ ]:
# Создайте или загрузите my_queries_df и опциональный my_schema_df здесь.
# Если DataFrame уже существуют в kernel, оставьте эту ячейку без изменений.


## 4. Получение входов и запуск анализа


In [ ]:
for state_name in (*NOTEBOOK_INPUT_STATE_NAMES, *NOTEBOOK_RESULT_STATE_NAMES):
    globals().pop(state_name, None)


def resolve_dataframe(variable_name, *, required_columns):
    if variable_name is None:
        return None
    if not isinstance(variable_name, str) or not variable_name.isidentifier():
        raise ValueError("DataFrame name must be a Python identifier or None.")
    if variable_name not in globals():
        raise NameError(f"DataFrame {variable_name!r} is not defined.")

    dataframe = globals()[variable_name]
    if not isinstance(dataframe, pd.DataFrame):
        raise TypeError(f"{variable_name!r} must be a pandas DataFrame.")

    missing = [name for name in required_columns if name not in dataframe.columns]
    if missing:
        raise ValueError(
            f"{variable_name!r} is missing required columns: {', '.join(missing)}"
        )
    return dataframe


def normalize_schema_dataframe(dataframe, *, variable_name):
    if dataframe is None:
        return None

    normalized = dataframe.copy(deep=True)
    schema_columns = [
        name
        for name in ("table_schema", "schema_name")
        if name in normalized.columns
    ]
    if not schema_columns:
        raise ValueError(
            f"{variable_name!r} needs schema_name or table_schema."
        )

    checked_columns = [*schema_columns, "table_name", "column_name"]
    for column in checked_columns:
        invalid_rows = [
            row_index
            for row_index, value in normalized[column].items()
            if not isinstance(value, str) or not value.strip()
        ]
        if invalid_rows:
            raise ValueError(
                f"{variable_name!r}.{column} contains empty or non-string "
                f"values at rows: {invalid_rows}"
            )
        normalized[column] = normalized[column].str.strip()

    if len(schema_columns) == 2:
        conflicts = normalized[
            normalized["table_schema"] != normalized["schema_name"]
        ]
        if not conflicts.empty:
            raise ValueError(
                f"{variable_name!r} has conflicting table_schema/schema_name "
                f"at rows: {conflicts.index.tolist()}"
            )
        normalized = normalized.drop(columns=["schema_name"])
    elif schema_columns == ["schema_name"]:
        normalized = normalized.rename(columns={"schema_name": "table_schema"})

    return normalized


def resolve_notebook_inputs():
    resolved_queries_df = resolve_dataframe(
        QUERY_DF_NAME,
        required_columns=("query_text", "query_text_template"),
    )
    resolved_schema_df = resolve_dataframe(
        SCHEMA_DF_NAME,
        required_columns=("table_name", "column_name"),
    )
    resolved_schema_df = normalize_schema_dataframe(
        resolved_schema_df,
        variable_name=SCHEMA_DF_NAME,
    )
    return resolved_queries_df, resolved_schema_df


input_queries_df, input_schema_df = resolve_notebook_inputs()
schema_df = input_schema_df


In [ ]:
for state_name in NOTEBOOK_RESULT_STATE_NAMES:
    globals().pop(state_name, None)


result = analyze_dataframe(
    input_queries_df,
    schema_df=input_schema_df,
    default_schema=DEFAULT_SCHEMA,
    output_dir=OUTPUT_DIR,
    build_html=BUILD_HTML,
    example_limit=20,
    source_label=QUERY_DF_NAME,
)

row_analysis_df = result.row_analysis_df
details_df = result.details_df
aggregate_df = result.aggregate_df
catalog_tables_df = result.catalog_tables_df
catalog_columns_df = result.catalog_columns_df
errors_df = result.errors_df

assert len(row_analysis_df) == len(input_queries_df)


## 5. Результаты

Главные результаты — `row_analysis_df`, `details_df` и `aggregate_df`.
Дополнительно доступны каталог таблиц/колонок и отдельный DataFrame ошибок.
HTML и JSON создаются только при заполненном `OUTPUT_DIR`; HTML также требует
`BUILD_HTML=True`.


In [ ]:
display(row_analysis_df)
display(details_df)
display(aggregate_df)
display(catalog_tables_df)
display(catalog_columns_df)
display(errors_df)

for artifact_name, artifact_path in result.artifact_paths.items():
    print(f"{artifact_name}: {artifact_path}")
